# CEO Letter / Annual Report Event Study — US Insurers (First Pass)

Tests whether abnormal stock returns around CEO letter/annual report publication dates correlate
with ESG narrative specificity. **US-region firms only for this first pass** (7 companies x 12
fiscal years = 84 firm-years). Kept **fully separate** from
`PHDp2_CEOLetters_AnnualReports_TextAnalysis.ipynb` and `PHDp2_FinancialData_Regression.ipynb` --
this notebook does not import from or modify either. Not validated yet -- developed on branch
`event-study`, not `main`.

**Report-only pipeline.** No merge with specificity measures, no regression -- Step 5 ends with a
confirmed firm-year event-study panel, full stop.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# pip installs yfinance if not already present in this Colab runtime
try:
    import yfinance as yf
except ImportError:
    %pip install -q yfinance
    import yfinance as yf

print("yfinance version:", yf.__version__)


## Step 1 — Load event dates

Read `Combined_Insurer_Publication_Dates.xlsx`, filter to `Region == "United States"`, confirm 84
firm-years (7 companies x 12 fiscal years) before proceeding. Chubb's rows already unify the
2016 ACE Limited -> Chubb Limited name change into a single `"Chubb"` company label in the source
file (confirmed directly: FY2012-2015 rows carry a note "Filed as ACE Limited pre-2016 merger,"
FY2016-2023 rows carry "Filed as Chubb Limited" -- both already grouped under `Company == "Chubb"`
with `Region == "United States"`), so no company-name unification step is needed here -- it's
already done in the source data. The ticker-level ACE/CB split is handled in Step 2/3 instead,
since that's a market-data concern, not an event-date concern.

In [ ]:
# Adjust this path if the file lives elsewhere in your Drive.
event_dates_path = "/content/drive/MyDrive/phd/Data/Combined_Insurer_Publication_Dates.xlsx"

df_dates_raw = pd.read_excel(event_dates_path)
print(f"Full file: {df_dates_raw.shape}")
print("Regions present:", sorted(df_dates_raw['Region'].unique()))

df_us = df_dates_raw[df_dates_raw["Region"] == "United States"].copy()
print(f"\nUS-region rows: {len(df_us)}")

# ------------------------------------------------------------
# Confirm 84 firm-years (7 companies x 12 fiscal years) before proceeding,
# per instruction -- stop and investigate if this doesn't hold, rather
# than silently continue on a wrong filter.
# ------------------------------------------------------------
n_companies = df_us["Company"].nunique()
n_years = df_us["Fiscal Year"].nunique()
print(f"Distinct companies: {n_companies}")
print(f"Distinct fiscal years: {n_years}")
print(f"Companies: {sorted(df_us['Company'].unique())}")
print(f"Fiscal years: {sorted(df_us['Fiscal Year'].unique())}")

counts_per_company = df_us["Company"].value_counts()
print("\nRows per company:")
print(counts_per_company)

assert len(df_us) == 84, f"Expected 84 US firm-years, got {len(df_us)} -- STOP, investigate before continuing."
assert n_companies == 7, f"Expected 7 US companies, got {n_companies}"
assert (counts_per_company == 12).all(), "Not every company has exactly 12 fiscal years -- STOP."
print("\nCONFIRMED: 84 US firm-years (7 companies x 12 fiscal years).")

# ------------------------------------------------------------
# Parse Publication Date (stored as a string in the source file, not a
# native Excel date) and sanity-check the Chubb/ACE transition boundary
# noted above.
# ------------------------------------------------------------
df_us["Publication Date"] = pd.to_datetime(df_us["Publication Date"])
print(f"\nPublication date range: {df_us['Publication Date'].min()} to {df_us['Publication Date'].max()}")

chubb = df_us[df_us["Company"] == "Chubb"][["Fiscal Year", "Publication Date", "Note"]].sort_values("Fiscal Year")
print("\nChubb rows (confirm ACE->Chubb transition sits between FY2015 and FY2016):")
print(chubb.to_string(index=False))


## Step 2 — Tickers

Company -> ticker mapping, plus the benchmark (`URTH`, iShares MSCI World ETF). Chubb needs
special handling: pre-2016 it traded as **ACE Limited** under ticker **ACE**, not `CB`. Rather
than assume where `CB`'s history starts, this pulls both `ACE` and `CB` and empirically checks
where each series actually starts/ends -- the actual data boundary, not an assumed merger-closing
date, decides the splice point (handled in Step 3).

In [ ]:
COMPANY_TICKER = {
    "American International Group (AIG)": "AIG",
    "Chubb": "CB",                     # current ticker; ACE spliced in for pre-merger history
    "MetLife, Inc.": "MET",
    "Prudential Financial, Inc.": "PRU",
    "The Allstate Corporation": "ALL",
    "The Progressive Corporation": "PGR",
    "The Travelers Companies, Inc.": "TRV",
}
ACE_TICKER = "ACE"          # Chubb's pre-2016-merger ticker (as ACE Limited)
BENCHMARK_TICKER = "URTH"   # iShares MSCI World ETF

# ACE is checked SEPARATELY and non-fatally below -- it's Chubb's
# pre-2016 ticker, and Yahoo Finance frequently drops a ticker's price
# history entirely after a symbol change/delisting rather than
# archiving it, so ACE failing to resolve at all (confirmed: fails even
# against a 2014 window, when it was definitely actively trading) is a
# real possible outcome, not necessarily a bug to "fix." The other 8
# tickers (7 companies' current symbols + URTH) are the ones that must
# all resolve for this pipeline to proceed at all.
required_tickers = list(COMPANY_TICKER.values()) + [BENCHMARK_TICKER]
print("Required tickers to verify:", required_tickers)

DEFAULT_PROBE_WINDOW = ("2019-01-01", "2019-01-31")

ticker_resolves = {}
for ticker in required_tickers:
    start, end = DEFAULT_PROBE_WINDOW
    probe = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    ok = len(probe) > 0
    ticker_resolves[ticker] = ok
    print(f"{ticker}: {'OK' if ok else 'FAILED'} ({len(probe)} rows in {start} probe window)")

failed = [t for t, ok in ticker_resolves.items() if not ok]
if failed:
    raise RuntimeError(f"These REQUIRED tickers did not resolve in yfinance: {failed} -- STOP, fix before Step 3.")
print("\nAll required tickers resolve.")

# ------------------------------------------------------------
# ACE probe (2014, when it was definitely trading) -- reported, not
# raised on, since the real question this pipeline needs answered is
# "does SOME source cover the pre-2016 Chubb event dates," and CB's own
# history (checked next) is the other candidate.
# ------------------------------------------------------------
ace_probe = yf.download("ACE", start="2014-01-01", end="2014-01-31", progress=False, auto_adjust=True)
ace_probe_ok = len(ace_probe) > 0
print(f"\nACE (informational, not required): {'OK' if ace_probe_ok else 'NO DATA'} "
      f"({len(ace_probe)} rows in 2014-01 probe window)")

# ------------------------------------------------------------
# Check whether CB's own history already extends back far enough to
# cover the earliest Chubb event dates (2012-2015 fiscal years, i.e.
# publication dates as early as 2013). Report the actual first
# available date rather than assume -- this determines whether the
# pre-2016 Chubb firm-years are usable at all via yfinance.
# ------------------------------------------------------------
cb_full = yf.download("CB", start="2011-01-01", end="2016-06-30", progress=False, auto_adjust=True)
ace_full = yf.download("ACE", start="2011-01-01", end="2016-06-30", progress=False, auto_adjust=True) if ace_probe_ok else None

print(f"\nCB: first available date = {cb_full.index.min() if len(cb_full) else 'NO DATA'}")
if ace_full is not None and len(ace_full):
    print(f"ACE: first available date = {ace_full.index.min()}, last available date = {ace_full.index.max()}")
else:
    print("ACE: NO DATA available from yfinance at all (confirmed via both the 2014 and "
          "2011-2016 pulls) -- Yahoo Finance has evidently dropped this ticker's history "
          "entirely rather than archiving it under the old symbol.")

earliest_chubb_event = chubb["Publication Date"].min()
print(f"\nEarliest Chubb event date requiring price history: {earliest_chubb_event}")
print(f"Estimation window for that event needs data back to roughly "
      f"{earliest_chubb_event - pd.Timedelta(days=380)} (250 trading days plus weekends/holidays buffer)")

CB_COVERS_PRE_2016 = len(cb_full) > 0 and cb_full.index.min() <= pd.Timestamp("2015-01-01")
ACE_AVAILABLE = ace_full is not None and len(ace_full) > 0

if CB_COVERS_PRE_2016:
    print("\nCB's own history extends back far enough -- no splice needed. VERIFY this isn't "
          "Yahoo silently truncating/misdating data rather than genuinely having ACE-era prices "
          "under the CB symbol, by spot-checking one known pre-2016 price point before trusting it.")
    CHUBB_STRATEGY = "cb_only"
elif ACE_AVAILABLE:
    print("\nCB does NOT cover pre-2016, but ACE does -- splicing ACE + CB (handled in Step 4).")
    CHUBB_STRATEGY = "splice"
else:
    print("\n" + "=" * 70)
    print("DATA GAP: neither CB nor ACE covers Chubb's pre-2016 fiscal years via yfinance.")
    print("=" * 70)
    print("This means Chubb FY2012-2015 (4 of the 84 firm-years) cannot be included in the "
          "event study through this data source. Two options, NOT decided here:")
    print("  (a) Drop those 4 Chubb firm-years, proceed with N=80 (84 - 4).")
    print("  (b) Source ACE's pre-2016 price history from a different provider (e.g. a paid "
          "data vendor, or a manually-compiled CSV) and splice it in manually.")
    print("Proceeding with CB-only for Chubb (FY2016-2023) and flagging FY2012-2015 as excluded "
          "in Step 4/5's output -- report back and we can revisit if (b) is wanted instead.")
    CHUBB_STRATEGY = "cb_only_with_gap"

print(f"\nCHUBB_STRATEGY = {CHUBB_STRATEGY!r}")


## Step 3 — Pull daily price data

All 7 tickers (+ `ACE` for the pre-2016 Chubb splice) + `URTH`, Jan 2011 (estimation-window
buffer before the earliest 2012 fiscal-year event) through Dec 2023. Raw pulls saved to disk
before any processing, so Step 4/5 can be re-run without re-hitting the API.

In [ ]:
PRICE_START = "2011-01-01"
# 2023-12-31 originally -- WRONG: FY2023 events are published in Q1/Q2
# 2024 (latest event date 2024-04-26), so that cutoff excluded every
# FY2023 event window and part of some FY2023 estimation windows too.
# Confirmed systematic across all 7 companies (not Chubb-specific) via
# the exclusion-accounting re-check in the report-only summary cell.
# Extended to give buffer past the latest event date's post-event window.
PRICE_END = "2024-07-31"

raw_prices = {}
tickers_to_pull = list(COMPANY_TICKER.values()) + [BENCHMARK_TICKER]
if CHUBB_STRATEGY == "splice":
    tickers_to_pull.append(ACE_TICKER)
else:
    print(f"CHUBB_STRATEGY = {CHUBB_STRATEGY!r} -- not pulling ACE (unavailable or unneeded).")

for ticker in tickers_to_pull:
    if ticker in raw_prices:
        continue  # CB only pulled once even though it's used for one company
    print(f"Pulling {ticker}...")
    data = yf.download(ticker, start=PRICE_START, end=PRICE_END, progress=False, auto_adjust=True)
    raw_prices[ticker] = data
    print(f"  {len(data)} rows, {data.index.min()} to {data.index.max()}")

raw_prices_dir = "/content/drive/MyDrive/phd/Data/event_study_raw_prices"
os.makedirs(raw_prices_dir, exist_ok=True)
for ticker, data in raw_prices.items():
    out_path = os.path.join(raw_prices_dir, f"{ticker}.csv")
    data.to_csv(out_path)
    print(f"Saved {ticker} -> {out_path}")

print(f"\nAll raw pulls saved to: {raw_prices_dir}")


## Step 4 — Market-model event study, per firm-year

Estimation window `[-250, -30]` trading days relative to the (trading-day-adjusted) event date;
OLS of firm return on `URTH` return -> firm-specific alpha, beta. Event windows: primary
`[-1, +1]`; also `[-2, +2]` and `[0, +1]` as robustness alternatives, all three reported.
Abnormal return = actual - (alpha + beta x URTH return); CAR = sum over the window, both signed
and `|CAR|` reported. Non-trading-day publication dates shift to the next available trading day
(reported how many). Firm-years with fewer than 100 valid estimation-window trading days excluded
(reported which, if any).

In [ ]:
# ------------------------------------------------------------
# Build one continuous return series per company. For Chubb, splice ACE
# (up to ACE's last available date) and CB (from CB's first available
# date) -- using the ACTUAL empirical data boundary from Step 2, not an
# assumed merger-closing date, so the estimation window for the FY2015
# event (published 2016-04-08, whose [-250,-30] estimation window
# extends back into 2015, i.e. BEFORE the ACE->CB ticker switch) is
# built from genuinely continuous, correctly-sourced price data rather
# than silently using CB data that doesn't exist yet for that period.
# ------------------------------------------------------------
def get_adj_close(ticker):
    # yfinance can return either flat columns or MultiIndex columns
    # (ticker as a sub-level) even for a single-ticker download,
    # depending on version -- handled robustly rather than assumed flat,
    # since a silent DataFrame-vs-Series mismatch here would break
    # everything downstream without an obvious error at the point of failure.
    close = raw_prices[ticker]["Close"]  # auto_adjust=True -> Close is already adjusted
    if isinstance(close, pd.DataFrame):
        close = close.squeeze("columns")
    return close.dropna()

def build_return_series(company):
    ticker = COMPANY_TICKER[company]
    if company == "Chubb" and CHUBB_STRATEGY == "splice":
        ace_close = get_adj_close(ACE_TICKER)
        cb_close = get_adj_close("CB")
        splice_date = cb_close.index.min()
        ace_part = ace_close[ace_close.index < splice_date]
        print(f"Chubb: splicing ACE (through {ace_part.index.max()}) "
              f"+ CB (from {cb_close.index.min()})")
        gap_days = (cb_close.index.min() - ace_part.index.max()).days
        if gap_days > 10:
            print(f"  WARNING: {gap_days}-day gap between ACE's last price and CB's first price "
                  f"-- investigate before trusting the spliced return series across this boundary.")
        combined_close = pd.concat([ace_part, cb_close]).sort_index()
        combined_close = combined_close[~combined_close.index.duplicated(keep="last")]
    elif company == "Chubb":
        # CHUBB_STRATEGY is "cb_only" or "cb_only_with_gap" -- CB history
        # only, no ACE splice. In the "_with_gap" case this means
        # FY2012-2015 events will have no estimation-window data
        # available and get excluded naturally by Step 4's existing
        # "estimation window falls outside available price history"
        # check below -- not special-cased here, just a consequence of
        # only having CB's actual price history to work with.
        combined_close = get_adj_close("CB")
        print(f"Chubb: CB only (CHUBB_STRATEGY={CHUBB_STRATEGY!r}), "
              f"history from {combined_close.index.min()}")
    else:
        combined_close = get_adj_close(ticker)
    returns = combined_close.pct_change().dropna()
    returns.name = company
    return returns

benchmark_returns = get_adj_close(BENCHMARK_TICKER).pct_change().dropna()
benchmark_returns.name = "URTH"

company_returns = {company: build_return_series(company) for company in COMPANY_TICKER}


In [ ]:
# ------------------------------------------------------------
# Trading-day calendar taken directly from URTH's own observed trading
# days (rather than a generic calendar), so "next available trading day"
# and window offsets are defined consistently with the actual price data
# being used.
# ------------------------------------------------------------
trading_days = benchmark_returns.index.sort_values()

def next_trading_day(date, calendar):
    idx = calendar.searchsorted(date)
    if idx >= len(calendar):
        return None
    return calendar[idx]

def trading_day_offset(date, calendar, offset):
    """Trading day `offset` sessions away from `date` (date must be IN calendar)."""
    pos = calendar.get_loc(date)
    new_pos = pos + offset
    if new_pos < 0 or new_pos >= len(calendar):
        return None
    return calendar[new_pos]

EVENT_WINDOWS = {
    "primary_-1_+1": (-1, 1),
    "robustness_-2_+2": (-2, 2),
    "robustness_0_+1": (0, 1),
    "robustness_0_+5": (0, 5),
    "robustness_0_+15": (0, 15),
    "robustness_0_+20": (0, 20),
    "robustness_0_+30": (0, 30),
    "robustness_0_+45": (0, 45),
    "robustness_0_+60": (0, 60),  # ~1 quarter -- tests delayed reaction to complex
                                   # narrative disclosure per the investor-distraction
                                   # literature (investors may process dense textual
                                   # content with a lag rather than instantly)
}
ESTIMATION_WINDOW = (-250, -30)
MIN_ESTIMATION_OBS = 100

results = []
n_shifted = 0
excluded_thin_estimation = []

for _, row in df_us.iterrows():
    company = row["Company"]
    fiscal_year = row["Fiscal Year"]
    raw_event_date = row["Publication Date"]

    firm_ret = company_returns[company]
    # returns are pct_change of price, indexed by the LATER of the two
    # price dates -- align against a calendar of dates where a return is
    # actually observable for this firm.
    firm_calendar = firm_ret.index

    # Shift to next available trading day if the raw event date isn't one.
    if raw_event_date in firm_calendar:
        event_date = raw_event_date
    else:
        event_date = next_trading_day(raw_event_date, firm_calendar)
        n_shifted += 1
        if event_date is None:
            print(f"SKIPPING {company} FY{fiscal_year}: no trading day found on/after "
                  f"{raw_event_date} in the available price history.")
            continue

    event_pos_check = firm_calendar.get_loc(event_date)

    # ---- estimation window ----
    est_start = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[0])
    est_end = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[1])
    if est_start is None or est_end is None:
        print(f"SKIPPING {company} FY{fiscal_year}: estimation window falls outside available "
              f"price history.")
        continue

    est_mask = (firm_ret.index >= est_start) & (firm_ret.index <= est_end)
    est_firm = firm_ret[est_mask]
    est_bench = benchmark_returns.reindex(est_firm.index).dropna()
    est_common = est_firm.index.intersection(est_bench.index)
    est_firm = est_firm.loc[est_common]
    est_bench = benchmark_returns.loc[est_common]

    n_est = len(est_common)
    if n_est < MIN_ESTIMATION_OBS:
        excluded_thin_estimation.append((company, fiscal_year, n_est))
        continue

    # OLS: firm_return = alpha + beta * bench_return
    X = np.column_stack([np.ones(n_est), est_bench.values])
    coefs, _, _, _ = np.linalg.lstsq(X, est_firm.values, rcond=None)
    alpha, beta = coefs[0], coefs[1]

    # ---- event windows ----
    car_values = {}
    for win_name, (lo, hi) in EVENT_WINDOWS.items():
        win_start = trading_day_offset(event_date, firm_calendar, lo)
        win_end = trading_day_offset(event_date, firm_calendar, hi)
        if win_start is None or win_end is None:
            car_values[win_name] = np.nan
            continue
        win_mask = (firm_ret.index >= win_start) & (firm_ret.index <= win_end)
        win_firm = firm_ret[win_mask]
        win_bench = benchmark_returns.reindex(win_firm.index)
        abnormal = win_firm - (alpha + beta * win_bench)
        car_values[win_name] = abnormal.sum()

    results.append({
        "Company": company,
        "Fiscal Year": fiscal_year,
        "Event Date (raw)": raw_event_date,
        "Event Date (adjusted)": event_date,
        "Shifted": event_date != raw_event_date,
        "Estimation N": n_est,
        "Alpha": alpha,
        "Beta": beta,
        "CAR[-1,+1]": car_values["primary_-1_+1"],
        "CAR[-2,+2]": car_values["robustness_-2_+2"],
        "CAR[0,+1]": car_values["robustness_0_+1"],
        "CAR[0,+5]": car_values["robustness_0_+5"],
        "CAR[0,+15]": car_values["robustness_0_+15"],
        "CAR[0,+20]": car_values["robustness_0_+20"],
        "CAR[0,+30]": car_values["robustness_0_+30"],
        "CAR[0,+45]": car_values["robustness_0_+45"],
        "CAR[0,+60]": car_values["robustness_0_+60"],
        "|CAR|[-1,+1]": abs(car_values["primary_-1_+1"]),
    })

print(f"Publication dates shifted to next trading day: {n_shifted} of {len(df_us)}")
print(f"\nFirm-years excluded for <{MIN_ESTIMATION_OBS} estimation-window observations: "
      f"{len(excluded_thin_estimation)}")
for company, fy, n in excluded_thin_estimation:
    print(f"  {company} FY{fy}: only {n} estimation-window observations")


## Step 5 — Firm-year event-study panel

`Company | Fiscal Year | Event Date (adjusted) | Estimation N | Alpha | Beta | CAR[-1,+1] |
CAR[-2,+2] | CAR[0,+1] | |CAR|[-1,+1]`. No merge with specificity measures, no regression -- this
panel, confirmed correct, is the end of this notebook's first pass.

In [ ]:
panel = pd.DataFrame(results)
panel = panel.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

print(f"Firm-years in final panel: {len(panel)} of {len(df_us)} original US firm-years")
print(f"({len(df_us) - len(panel)} excluded -- see Step 4's exclusion list above for why)")

display_cols = ["Company", "Fiscal Year", "Event Date (adjusted)", "Estimation N",
                "Alpha", "Beta", "CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]",
                "CAR[0,+5]", "CAR[0,+15]", "CAR[0,+20]", "CAR[0,+30]", "CAR[0,+45]",
                "CAR[0,+60]", "|CAR|[-1,+1]"]
display(panel[display_cols])

output_path = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_US.csv"
panel[display_cols].to_csv(output_path, index=False)
print(f"\nSaved panel to: {output_path}")


## Report-only summary

CAR summary stats, exclusions and why, confirmation of the Chubb/ACE handling, and a spot-check
for unadjusted stock splits (an obviously-wrong single-day return, e.g. close to -50% or +100%,
is the classic signature of a split that `auto_adjust=True` failed to correctly adjust for --
checked explicitly below rather than assumed fine).

In [ ]:
print("=" * 70)
print("CAR summary statistics")
print("=" * 70)
print(panel[["CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+15]", "CAR[0,+20]",
             "CAR[0,+30]", "CAR[0,+45]", "CAR[0,+60]", "|CAR|[-1,+1]"]].describe())

print(f"\n{'=' * 70}\nExclusions\n{'=' * 70}")
print(f"Original US firm-years: {len(df_us)}")
print(f"Excluded for <{MIN_ESTIMATION_OBS} estimation obs: {len(excluded_thin_estimation)}")
for company, fy, n in excluded_thin_estimation:
    print(f"  {company} FY{fy}: {n} obs")
print(f"Final panel: {len(panel)}")

# ------------------------------------------------------------
# GENERAL exclusion-accounting check, all 7 companies, not just Chubb --
# this is what actually caught the FY2023/PRICE_END bug (every company
# was silently missing exactly FY2023, not something the Chubb-specific
# check below would have caught on its own). Re-run every time this
# notebook runs, not just after a known bug, since a systematic gap like
# this produces no error/warning on its own -- rows just don't show up.
# ------------------------------------------------------------
print(f"\n{'=' * 70}\nFull exclusion accounting -- every company, every fiscal year\n{'=' * 70}")
expected_pairs = set(zip(df_us["Company"], df_us["Fiscal Year"]))
actual_pairs = set(zip(panel["Company"], panel["Fiscal Year"]))
missing_pairs = sorted(expected_pairs - actual_pairs)
unexpected_pairs = sorted(actual_pairs - expected_pairs)

print(f"Expected firm-years: {len(expected_pairs)}")
print(f"Present in final panel: {len(actual_pairs)}")

if missing_pairs:
    print(f"\nMISSING firm-years ({len(missing_pairs)}):")
    missing_by_company = {}
    for company, fy in missing_pairs:
        missing_by_company.setdefault(company, []).append(fy)
    for company, fys in missing_by_company.items():
        print(f"  {company}: {sorted(fys)}")
    # Flag whether the gap is isolated to one company (e.g. the known
    # Chubb/ACE gap) or systematic across companies (e.g. the PRICE_END
    # bug this replaced) -- these need different explanations.
    companies_affected = len(missing_by_company)
    if companies_affected > 1:
        common_fys = set.intersection(*[set(fys) for fys in missing_by_company.values()])
        if common_fys:
            print(f"\n  SYSTEMATIC: {companies_affected} companies all missing fiscal year(s) "
                  f"{sorted(common_fys)} -- check PRICE_END / event-date coverage, not a "
                  f"single-company data issue.")
else:
    print("\nNo missing firm-years.")

if unexpected_pairs:
    print(f"\nUNEXPECTED firm-years in panel not in source data ({len(unexpected_pairs)}): "
          f"{unexpected_pairs} -- investigate, this should not happen.")

assert len(panel) == 84 or (len(panel) == 80 and CHUBB_STRATEGY == "cb_only_with_gap"), (
    f"Final panel has {len(panel)} rows -- expected 84 (full) or 80 (with the known, reported "
    f"Chubb FY2012-2015 gap). Anything else means an unexplained loss -- STOP and investigate "
    f"before trusting this panel."
)
if len(panel) == 84:
    print("\nCONFIRMED: 84/84 firm-years present, no gaps.")
else:
    print(f"\nCONFIRMED: {len(panel)}/84 firm-years present, with the expected and reported "
          f"Chubb FY2012-2015 gap (CHUBB_STRATEGY={CHUBB_STRATEGY!r}) -- no OTHER unexplained gap.")

print(f"\n{'=' * 70}\nChubb / ACE ticker transition\n{'=' * 70}")
print(f"CHUBB_STRATEGY = {CHUBB_STRATEGY!r}")
chubb_panel = panel[panel["Company"] == "Chubb"]
chubb_fiscal_years_in_panel = sorted(chubb_panel["Fiscal Year"].tolist())
chubb_fiscal_years_expected = sorted(df_us[df_us["Company"] == "Chubb"]["Fiscal Year"].tolist())
chubb_missing = sorted(set(chubb_fiscal_years_expected) - set(chubb_fiscal_years_in_panel))
print(f"Chubb fiscal years in final panel: {chubb_fiscal_years_in_panel}")
if chubb_missing:
    print(f"Chubb fiscal years MISSING from final panel: {chubb_missing} "
          f"({'expected -- yfinance has no ACE-era price data, see Step 2' if CHUBB_STRATEGY == 'cb_only_with_gap' else 'unexpected, investigate'})")
else:
    print("All 12 Chubb fiscal years present.")
print(chubb_panel[["Fiscal Year", "Event Date (adjusted)"]].to_string(index=False))
if CHUBB_STRATEGY == "splice":
    print("\nSee Step 4's splice-point print for the exact ACE-end / CB-start dates used, and "
          "whether the gap-check warning fired.")
elif CHUBB_STRATEGY == "cb_only_with_gap":
    print("\nNeither CB nor ACE covers Chubb's pre-2016 history via yfinance (confirmed in "
          "Step 2) -- Chubb FY2012-2015 are excluded from this panel, not spliced or imputed. "
          "If those 4 firm-years are needed, pre-2016 ACE Limited prices would need to be "
          "sourced from a different data provider.")

print(f"\n{'=' * 70}\nSpot-check: unadjusted stock splits\n{'=' * 70}")
print("Flags any single-day return whose magnitude exceeds 20% for any of the 7 firms -- a very "
      "large one-day move that isn't obviously tied to a known market-wide shock is the classic "
      "signature of an unadjusted split slipping through (auto_adjust=True should prevent this, "
      "but checked explicitly rather than assumed).")
for company, ret in company_returns.items():
    extreme = ret[ret.abs() > 0.20]
    if len(extreme) > 0:
        print(f"\n{company}: {len(extreme)} day(s) with |return| > 20%:")
        print(extreme.to_string())
    else:
        print(f"{company}: none")


---
# Part 2 — CAR ~ Specificity + Sentiment (Extension)

Extends the Part 1 event-study panel with CEO-letter specificity and sentiment measures, and
tests whether they're associated with abnormal returns around the publication event. All data
loaded directly from Drive (same convention as the rest of this project) -- no manual uploads.

**`Specificity_{i,t}`** and **`ESGSentiment_{i,t}`** use the same **Corporate + tier-1-plus
restriction** as `Specificity_mean`/`ESGSentiment_Corporate` elsewhere in this project (confirmed
choice, not a default). **`Sentiment_{i,t}`** (overall, not ESG-restricted) uses ALL sentences
from the FinBERT file, deliberately not ESG-filtered -- it's meant to capture the letter's general
tone, not just its ESG content.

Kept in the same notebook as Part 1 (not a separate file) since it needs the just-computed
event-study panel directly. Interpretation is deliberately conservative throughout: *associated
with*, never *causes*.

## Step 6 — Merge event-study panel with specificity and sentiment data

Three sources, all from Drive:
1. `ceo_letter_event_study_panel_US.csv` (Part 1's output, 84 firm-years)
2. `df_ar_ceo_sentences_esg_combined_specificity.csv` (sentence-level, ESG-flagged sentences only,
   already carries `is_corporate_tier1plus`/`tier_if_corp_t1p`/etc.)
3. `df_ar_ceo_sentences_finbert_senti.csv` (sentence-level, ALL sentences, for overall sentiment)

Merge key: `Company` (harmonized -- the event-study panel uses the full legal names from
`Combined_Insurer_Publication_Dates.xlsx`, the CEO-letter files use the project's existing short
names) + `Fiscal Year` / `Year`. Reports matched observations, missing years, and duplicates
before building anything on top of this merge.

In [ ]:
# ============================================================
# Step 6.1 -- Load all three sources from Drive
# ============================================================
# Extended from the US-only panel to the full 22-firm combined panel
# (built in Part 3, Step 13) now that the non-US event-study data has
# been validated.
event_panel_path = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_full22.csv"
specificity_path = "/content/drive/MyDrive/phd/Data/df_ar_ceo_sentences_esg_combined_specificity.csv"
finbert_path = "/content/drive/MyDrive/phd/Data/df_ar_ceo_sentences_finbert_senti.csv"

event_panel = pd.read_csv(event_panel_path)
event_panel["Event Date (adjusted)"] = pd.to_datetime(event_panel["Event Date (adjusted)"])
print(f"Event-study panel: {event_panel.shape}")

df_specificity = pd.read_csv(specificity_path)
print(f"Specificity (sentence-level, ESG-flagged only): {df_specificity.shape}")

df_finbert = pd.read_csv(finbert_path)
print(f"FinBERT sentiment (sentence-level, ALL sentences): {df_finbert.shape}")

# ------------------------------------------------------------
# Company-name harmonization. The event-study panel uses the full legal
# names from Combined_Insurer_Publication_Dates.xlsx; the CEO-letter
# pipeline files use this project's existing short names (same raw
# names df_ar_cluster.csv itself is built from, before that notebook's
# own name_fixes remapping). Printed explicitly below rather than
# assumed silently correct -- if a name doesn't match, it will show up
# as 0 matched rows for that company and needs fixing here.
# ------------------------------------------------------------
print("\nEvent-study panel company names:", sorted(event_panel["Company"].unique()))
print("Specificity file company names:", sorted(df_specificity["Company Name"].unique()))
print("FinBERT file company names:", sorted(df_finbert["Company Name"].unique()))

# Extended to all 22 companies -- confirmed directly against the actual
# "Specificity file company names" / "FinBERT file company names"
# output above (both files use identical short names), not guessed.
SHORT_TO_LONG_NAME = {
    # US (original 7)
    "AIG": "American International Group (AIG)",
    "Chubb": "Chubb",
    "MET": "MetLife, Inc.",
    "MetLife": "MetLife, Inc.",
    "Prudential Financials": "Prudential Financial, Inc.",
    "Prudentials": "Prudential Financial, Inc.",
    "Allstate": "The Allstate Corporation",
    "Progressive": "The Progressive Corporation",
    "Travelers": "The Travelers Companies, Inc.",
    # Non-US (15)
    "AXA": "AXA SA",
    "Allianz": "Allianz SE",
    "China Life": "China Life Insurance Company Limited",
    "Daichi": "Dai-ichi Life Holdings",
    "Generali": "Assicurazioni Generali",
    "MSAD": "MS&AD Insurance Group Holdings",
    "Munich RE": "Munich Re",
    "PIC": "People\'s Insurance Company of China (PICC)",
    "Ping An": "Ping An Insurance Group",
    "Power Corp Canada": "Power Corporation of Canada",
    "Sompo": "Sompo Holdings",
    "Swiss RE": "Swiss Re",
    "Talanx": "Talanx AG",
    "Tokio Marine": "Tokio Marine Holdings",
    "Zurich": "Zurich Insurance Group",
}

df_specificity["Company"] = df_specificity["Company Name"].map(SHORT_TO_LONG_NAME)
df_finbert["Company"] = df_finbert["Company Name"].map(SHORT_TO_LONG_NAME)

unmapped_spec = df_specificity[df_specificity["Company"].isna()]["Company Name"].unique()
unmapped_finbert = df_finbert[df_finbert["Company"].isna()]["Company Name"].unique()
print(f"\nUnmapped company names in specificity file (not in our 7-company mapping -- expected, "
      f"this file covers the full 22-firm panel): {sorted(unmapped_spec)}")
print(f"Unmapped company names in FinBERT file (same expectation): {sorted(unmapped_finbert)}")

# Restrict both sentence-level files to the 22 companies actually in
# the event study, using the harmonized name -- and confirm all 22
# mapped successfully. SHORT_TO_LONG_NAME above currently only covers
# the 7 US companies -- the print statements right below list every
# actual company-name string in the specificity/FinBERT files, needed
# to extend the mapping to the 15 non-US companies without guessing
# spellings (a guessed-wrong short name fails silently as 0 matched
# rows, not an error).
panel_companies = set(event_panel["Company"].unique())
df_specificity_panel = df_specificity[df_specificity["Company"].isin(panel_companies)].copy()
df_finbert_panel = df_finbert[df_finbert["Company"].isin(panel_companies)].copy()

mapped_companies_spec = set(df_specificity_panel["Company"].unique())
mapped_companies_finbert = set(df_finbert_panel["Company"].unique())
print(f"\nPanel companies ({len(panel_companies)}) with NO rows at all in specificity file after "
      f"mapping: {sorted(panel_companies - mapped_companies_spec)}")
print(f"Panel companies with NO rows at all in FinBERT file after mapping: "
      f"{sorted(panel_companies - mapped_companies_finbert)}")

# Aliases used later in this notebook (Steps 6.2+) -- kept so the rest
# of Part 2's code doesn't need to change now that this covers all 22
# companies rather than just the 7 US ones.
df_specificity_us = df_specificity_panel
df_finbert_us = df_finbert_panel


In [ ]:
# ============================================================
# Step 6.2 -- Firm-year aggregation: Specificity_{i,t} and
# ESGSentiment_{i,t} (Corporate + tier-1-plus restricted, matching
# Specificity_mean / ESGSentiment_Corporate elsewhere in this project).
# Mirrors the exact same aggregation logic already used there --
# is_corporate_tier1plus / tier_if_corp_t1p / is_corp_t1p_positive /
# is_corp_t1p_negative are already-computed columns in this file, not
# re-derived differently here.
# ============================================================
spec_fy = (
    df_specificity_us
    .groupby(["Company", "Year"])
    .agg(
        n_corporate_tier1plus=("is_corporate_tier1plus", "sum"),
        Specificity=("tier_if_corp_t1p", "mean"),
        n_corp_t1p_positive=("is_corp_t1p_positive", "sum"),
        n_corp_t1p_negative=("is_corp_t1p_negative", "sum"),
    )
    .reset_index()
)
spec_fy["ESGSentiment"] = np.where(
    spec_fy["n_corporate_tier1plus"] > 0,
    (spec_fy["n_corp_t1p_positive"] - spec_fy["n_corp_t1p_negative"]) / spec_fy["n_corporate_tier1plus"],
    np.nan,
)
spec_fy = spec_fy.rename(columns={"Year": "Fiscal Year"})
print(f"Specificity/ESGSentiment firm-years built: {len(spec_fy)}")
print(f"Firm-years with zero Corporate+tier1+ sentences (Specificity/ESGSentiment undefined, "
      f"NOT imputed): {(spec_fy['n_corporate_tier1plus'] == 0).sum()}")
display(spec_fy.head())

# ============================================================
# Step 6.3 -- Firm-year aggregation: Sentiment_{i,t} (overall, ALL
# sentences, not ESG-restricted). df_ar_ceo_sentences_finbert_senti.csv
# carries sent_label per sentence regardless of ESG relevance.
# ============================================================
df_finbert_us["sent_label"] = df_finbert_us["sent_label"].astype(str).str.lower().str.strip()
df_finbert_us["is_positive"] = (df_finbert_us["sent_label"] == "positive").astype(int)
df_finbert_us["is_negative"] = (df_finbert_us["sent_label"] == "negative").astype(int)

senti_fy = (
    df_finbert_us
    .groupby(["Company", "Year"])
    .agg(
        total_sentences=("Sentence", "count"),
        n_positive=("is_positive", "sum"),
        n_negative=("is_negative", "sum"),
    )
    .reset_index()
)
senti_fy["Sentiment"] = (senti_fy["n_positive"] - senti_fy["n_negative"]) / senti_fy["total_sentences"]
senti_fy = senti_fy.rename(columns={"Year": "Fiscal Year"})
print(f"\nOverall sentiment firm-years built: {len(senti_fy)}")
display(senti_fy.head())


In [ ]:
# ============================================================
# Step 6.4 -- Merge everything into the analysis panel, and report
# matched/missing/duplicates explicitly before building anything else.
# ============================================================
analysis = event_panel.merge(
    spec_fy[["Company", "Fiscal Year", "n_corporate_tier1plus", "Specificity", "ESGSentiment"]],
    on=["Company", "Fiscal Year"], how="left"
)
analysis = analysis.merge(
    senti_fy[["Company", "Fiscal Year", "total_sentences", "Sentiment"]],
    on=["Company", "Fiscal Year"], how="left"
)

print(f"Event-study panel rows: {len(event_panel)}")
print(f"Analysis panel rows after merge: {len(analysis)} (should still be 84 -- a left merge "
      f"should not change row count)")
assert len(analysis) == len(event_panel), "Row count changed after merge -- STOP, check for duplicate keys."

n_missing_specificity = analysis["Specificity"].isna().sum()
n_missing_esgsentiment = analysis["ESGSentiment"].isna().sum()
n_missing_sentiment = analysis["Sentiment"].isna().sum()
print(f"\nMissing Specificity (zero Corporate+tier1+ sentences that firm-year): {n_missing_specificity}")
print(f"Missing ESGSentiment (same reason): {n_missing_esgsentiment}")
print(f"Missing overall Sentiment (firm-year not found in FinBERT file at all): {n_missing_sentiment}")

if n_missing_specificity > 0:
    print("\nFirm-years missing Specificity:")
    print(analysis.loc[analysis["Specificity"].isna(), ["Company", "Fiscal Year"]].to_string(index=False))
if n_missing_sentiment > 0:
    print("\nFirm-years missing overall Sentiment:")
    print(analysis.loc[analysis["Sentiment"].isna(), ["Company", "Fiscal Year"]].to_string(index=False))

# Duplicate check -- should be impossible given event_panel is one row
# per (Company, Fiscal Year) and both aggregations are grouped the same
# way, but checked explicitly rather than assumed.
dup_check = analysis.duplicated(subset=["Company", "Fiscal Year"]).sum()
print(f"\nDuplicate (Company, Fiscal Year) rows in analysis panel: {dup_check} (should be 0)")

print(f"\nMatched (non-missing Specificity AND Sentiment) firm-years for regression: "
      f"{((~analysis['Specificity'].isna()) & (~analysis['Sentiment'].isna())).sum()} of {len(analysis)}")


### Step 6.5 — Diagnose the missing firm-years (concentrated, not scattered)

Now that Step 6.1 covers all 22 companies, the missing-Specificity count jumped to 178 of 240 --
striking pattern: the 4 Japanese companies (Dai-ichi, MS&AD, Sompo, Tokio Marine) are only
PARTIALLY missing (consistent with the original 7-US-firm pattern of scattered "no qualifying
sentences that year" gaps), but every one of the other 11 non-US companies (Canada, China/HK x3,
France, Germany x3, Italy, Switzerland x2) is missing 100% of its firm-years with zero exceptions.
That is not what a scattered content gap looks like -- it looks like the Corporate-relevance/
specificity GPT classification pass may never have been run for those 11 companies at all. The
diagnostic below checks this directly per company-year rather than assuming it.

This checks, for exactly the missing (Company, Fiscal Year) pairs, whether the underlying
sentence-level files have: (a) no rows at all for that company-year (a source-corpus gap -- an
annual report/CEO letter that was never captured, consistent with the ALREADY-KNOWN gap-years
documented elsewhere in this project, e.g. Progressive 2020, Power Corp Canada 2022, Talanx 2020),
(b) rows present but zero ESG-flagged/Corporate+tier1+ sentences (a genuine content finding, not a
bug), or (c) something else entirely (a real remaining bug).

In [ ]:
# ============================================================
# Step 6.5 -- Row-count diagnostic for the missing firm-years
# ============================================================
missing_specificity_pairs = analysis.loc[analysis["Specificity"].isna(), ["Company", "Fiscal Year"]]
missing_sentiment_pairs = analysis.loc[analysis["Sentiment"].isna(), ["Company", "Fiscal Year"]]

diag_rows = []
for _, row in missing_specificity_pairs.iterrows():
    company, fy = row["Company"], row["Fiscal Year"]
    spec_rows = df_specificity_us[(df_specificity_us["Company"] == company) & (df_specificity_us["Year"] == fy)]
    n_spec_rows = len(spec_rows)
    n_tier1plus = spec_rows["is_corporate_tier1plus"].sum() if n_spec_rows else 0
    finbert_rows = df_finbert_us[(df_finbert_us["Company"] == company) & (df_finbert_us["Year"] == fy)]
    n_finbert_rows = len(finbert_rows)

    if n_spec_rows == 0 and n_finbert_rows == 0:
        cause = "SOURCE GAP"
    elif n_spec_rows == 0 and n_finbert_rows > 0:
        cause = "MISMATCH"
    elif n_spec_rows > 0 and n_tier1plus == 0:
        cause = "genuine (0 tier1+)"
    else:
        cause = "UNEXPECTED"

    diag_rows.append({
        "Company": company, "Fiscal Year": fy, "spec_us_rows": n_spec_rows,
        "spec_us_tier1plus": int(n_tier1plus), "finbert_us_rows": n_finbert_rows, "cause": cause,
    })

diag_df = pd.DataFrame(diag_rows)

# 178 rows is too long to print reliably in one go (Colab silently
# truncates large text output without always flagging it) -- saved in
# full to Drive, and summarized here as a per-company x cause crosstab,
# which is compact regardless of row count and is what actually answers
# "is this a real 11-company classification gap."
diag_output_path = "/content/drive/MyDrive/phd/Data/missing_specificity_diagnostic_full22.csv"
diag_df.to_csv(diag_output_path, index=False)
print(f"Full {len(diag_df)}-row diagnostic saved to: {diag_output_path}")

print(f"\n{'=' * 70}\nPer-company x cause crosstab (counts of missing firm-years)\n{'=' * 70}")
crosstab = pd.crosstab(diag_df["Company"], diag_df["cause"])
crosstab["TOTAL missing"] = crosstab.sum(axis=1)
display(crosstab)

print(f"\n{'=' * 70}\nOverall cause totals across all {len(diag_df)} missing firm-years\n{'=' * 70}")
print(diag_df["cause"].value_counts().to_string())

print(f"\n{'=' * 70}\nFirm-years missing Sentiment specifically (not overlapping the above)\n{'=' * 70}")
sentiment_only_missing = missing_sentiment_pairs.merge(
    missing_specificity_pairs, on=["Company", "Fiscal Year"], how="left", indicator=True
)
sentiment_only_missing = sentiment_only_missing[sentiment_only_missing["_merge"] == "left_only"]
print(f"Firm-years missing Sentiment but NOT Specificity: {len(sentiment_only_missing)}")
for _, row in sentiment_only_missing.iterrows():
    company, fy = row["Company"], row["Fiscal Year"]
    finbert_rows = df_finbert_us[(df_finbert_us["Company"] == company) & (df_finbert_us["Year"] == fy)]
    print(f"  {company} FY{fy}: {len(finbert_rows)} FinBERT rows")

# ------------------------------------------------------------
# Also directly check: does the RAW (unfiltered-to-US) specificity/
# FinBERT file have any row at all under a DIFFERENT spelling for
# these company-years -- i.e. did SHORT_TO_LONG_NAME miss a spelling
# variant used only in certain years?
# ------------------------------------------------------------
print(f"\n{'=' * 70}\nAll distinct raw \'Company Name\' spellings ever used for MetLife/Prudential/"
      f"Progressive in the two source files (checking for a year-specific spelling variant)\n{'=' * 70}")
for keyword in ["Met", "Prudential", "Progressive"]:
    spec_variants = df_specificity[df_specificity["Company Name"].str.contains(keyword, case=False, na=False)]["Company Name"].unique()
    finbert_variants = df_finbert[df_finbert["Company Name"].str.contains(keyword, case=False, na=False)]["Company Name"].unique()
    print(f"\n\'{keyword}\' -- specificity file: {sorted(spec_variants)}")
    print(f"\'{keyword}\' -- FinBERT file: {sorted(finbert_variants)}")


## Step 7 — Complete-case sample, descriptive statistics, correlations

Regression sample = firm-years with both `Specificity`/`ESGSentiment` and `Sentiment` defined
(non-imputed; a firm-year with zero Corporate+tier1+ sentences is dropped, not filled with 0 or
a mean). Tables 1 and 2 as specified.

In [ ]:
# ============================================================
# Step 7.1 -- Complete-case regression sample
# ============================================================
reg_cols = ["Specificity", "ESGSentiment", "Sentiment", "CAR[-1,+1]", "|CAR|[-1,+1]"]
reg_sample = analysis.dropna(subset=reg_cols).copy()
print(f"Complete-case regression sample: {len(reg_sample)} of {len(analysis)} firm-years")
print(f"Dropped: {len(analysis) - len(reg_sample)} "
      f"({analysis[~analysis.index.isin(reg_sample.index)][['Company', 'Fiscal Year']].to_string(index=False) if len(analysis) > len(reg_sample) else 'none'})")

# ============================================================
# Step 7.2 -- Table 1: descriptive statistics
# ============================================================
table1_cols = ["CAR[-1,+1]", "|CAR|[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]", "Specificity", "ESGSentiment", "Sentiment"]
table1_cols = [c for c in table1_cols if c in reg_sample.columns]
table1 = reg_sample[table1_cols].agg(["mean", "std", "min", "max", "count"]).T
print("Table 1 -- Descriptive statistics")
display(table1.round(4))

# ============================================================
# Step 7.3 -- Table 2: correlation matrix
# ============================================================
table2_cols = ["CAR[-1,+1]", "|CAR|[-1,+1]", "Specificity", "ESGSentiment", "Sentiment"]
table2 = reg_sample[table2_cols].corr()
print("\nTable 2 -- Correlation matrix")
display(table2.round(3))


## Step 8 — Regression models

Four model specifications (Specificity only; Sentiment only; Specificity + Sentiment; + interaction),
each under three fixed-effects specifications (pooled OLS; firm FE; firm + year FE). Primary DV
`CAR[-1,+1]` and `|CAR|[-1,+1]`; robustness windows (`CAR[-2,+2]`, `CAR[0,+1]`, `CAR[0,+5]`,
`CAR[0,+20]`, `CAR[0,+60]` if present) reported separately below, not repeated across all 4x3 combinations to
keep the primary table readable.

**Small-sample caveat, stated once rather than repeated on every table below:** N is at most 240
(likely fewer after the complete-case restriction and after the exclusions documented in Part 3),
across up to 22 firms. Firm+year FE with up to 22 firms and 12 years still uses a large share of
the available degrees of freedom -- coefficients and SEs from Specification C should be read as
exploratory, not as precision estimates on the scale a much larger panel would give. Firm-clustered
SEs with up to 22 clusters are closer to, but still below, the ~30-50 cluster rule of thumb for
reliable cluster-robust asymptotics (better-founded than the original 7-firm US-only pass, not
fully reliable); reported anyway (as requested) alongside HC3, with that caveat attached rather
than silently omitted.

In [ ]:
# ============================================================
# Step 8.1 -- Fit all Model x Specification combinations for a given DV
# ============================================================
import statsmodels.formula.api as smf

MODEL_FORMULAS = {
    "Model 1 (Specificity only)": "Specificity",
    "Model 2 (Sentiment only)": "Sentiment",
    "Model 3 (Specificity + Sentiment)": "Specificity + Sentiment",
    "Model 4 (+ interaction)": "Specificity + Sentiment + Specificity:Sentiment",
}

FE_SPECS = {
    "A: Pooled OLS": "",
    "B: Firm FE": " + C(Company)",
    "C: Firm + Year FE": " + C(Company) + C(Q(\'Fiscal Year\'))",
}

def fit_all_models(df, dv, cov_type="HC3", cov_kwds=None):
    out = {}
    for model_label, rhs in MODEL_FORMULAS.items():
        for fe_label, fe_terms in FE_SPECS.items():
            formula = f"Q(\'{dv}\') ~ {rhs}{fe_terms}"
            try:
                kwargs = {"cov_type": cov_type}
                if cov_kwds is not None:
                    kwargs["cov_kwds"] = cov_kwds
                m = smf.ols(formula=formula, data=df).fit(**kwargs)
                out[(model_label, fe_label)] = m
            except Exception as e:
                print(f"FAILED: {model_label} / {fe_label} / {dv}: {e}")
    return out

models_car_hc3 = fit_all_models(reg_sample, "CAR[-1,+1]", cov_type="HC3")
models_abscar_hc3 = fit_all_models(reg_sample, "|CAR|[-1,+1]", cov_type="HC3")

models_car_cluster = fit_all_models(
    reg_sample, "CAR[-1,+1]", cov_type="cluster", cov_kwds={"groups": reg_sample["Company"]}
)
models_abscar_cluster = fit_all_models(
    reg_sample, "|CAR|[-1,+1]", cov_type="cluster", cov_kwds={"groups": reg_sample["Company"]}
)

print(f"Fitted {len(models_car_hc3)} model x FE-spec combinations for CAR[-1,+1] (HC3)")
print(f"Fitted {len(models_abscar_hc3)} model x FE-spec combinations for |CAR|[-1,+1] (HC3)")


In [ ]:
# ============================================================
# Step 8.2 -- Table 3: main results, Model 3 and 4, HC3 and cluster SEs,
# all three FE specs, both DVs.
# ============================================================
def summarize_model(m, terms):
    rows = []
    for term in terms:
        if term not in m.params.index:
            continue
        rows.append({
            "term": term, "coef": m.params[term], "se": m.bse[term], "p": m.pvalues[term],
            "N": int(m.nobs), "R2": m.rsquared,
        })
    return rows

table3_rows = []
key_terms = ["Specificity", "Sentiment", "Specificity:Sentiment"]

for dv_label, models_hc3, models_cluster in [
    ("CAR[-1,+1]", models_car_hc3, models_car_cluster),
    ("|CAR|[-1,+1]", models_abscar_hc3, models_abscar_cluster),
]:
    for model_label in ["Model 3 (Specificity + Sentiment)", "Model 4 (+ interaction)"]:
        for fe_label in FE_SPECS:
            key = (model_label, fe_label)
            if key not in models_hc3:
                continue
            for row in summarize_model(models_hc3[key], key_terms):
                row.update({"dv": dv_label, "model": model_label, "fe": fe_label, "se_type": "HC3"})
                table3_rows.append(row)
            for row in summarize_model(models_cluster[key], key_terms):
                row.update({"dv": dv_label, "model": model_label, "fe": fe_label, "se_type": "cluster(firm)"})
                table3_rows.append(row)

table3 = pd.DataFrame(table3_rows)
table3["sig"] = table3["p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "")

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
print("Table 3 -- Main regression results (Models 3-4, all FE specs, HC3 and cluster SEs)")
display(table3[["dv", "model", "fe", "se_type", "term", "coef", "se", "p", "sig", "N", "R2"]].round(4))

# ------------------------------------------------------------
# Models 1-2 (single-regressor) reported separately, more compactly --
# these establish the baseline before Models 3-4's combined/interaction
# story.
# ------------------------------------------------------------
table_m1m2_rows = []
for dv_label, models_hc3 in [("CAR[-1,+1]", models_car_hc3), ("|CAR|[-1,+1]", models_abscar_hc3)]:
    for model_label in ["Model 1 (Specificity only)", "Model 2 (Sentiment only)"]:
        for fe_label in FE_SPECS:
            key = (model_label, fe_label)
            if key not in models_hc3:
                continue
            for row in summarize_model(models_hc3[key], key_terms):
                row.update({"dv": dv_label, "model": model_label, "fe": fe_label})
                table_m1m2_rows.append(row)
table_m1m2 = pd.DataFrame(table_m1m2_rows)
table_m1m2["sig"] = table_m1m2["p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "")
print("\nModels 1-2 (single-regressor baselines, HC3 SEs)")
display(table_m1m2[["dv", "model", "fe", "term", "coef", "se", "p", "sig", "N", "R2"]].round(4))


## Step 9 — Robustness

1. Alternative CAR windows (Model 3, Specification C only, to keep this compact).
2. Influential observations: Cook's distance and studentized residuals on the primary Model 3 /
   Spec C / `CAR[-1,+1]` fit, with AIG FY2019 and any COVID-period (2020) event specifically
   called out, plus the fit re-estimated excluding whatever the data actually flags (not
   pre-assumed to be those specific points).
3. Wild cluster bootstrap: **attempted, with an explicit caveat**. With up to 22 firms, cluster-
   robust inference (HC3-cluster or a full wild bootstrap alike) is still below the conventional
   30-50-cluster guidance for reliable asymptotics, though closer than the original 7-firm US-only
   pass -- reported for completeness, not because 22 clusters makes this fully reliable.

In [ ]:
# ============================================================
# Step 9.1 -- Alternative CAR windows, Model 3 / Spec C only
# ============================================================
alt_windows = ["CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]"]
for win_col in ["CAR[0,+5]", "CAR[0,+15]", "CAR[0,+20]", "CAR[0,+30]", "CAR[0,+45]", "CAR[0,+60]"]:
    if win_col in reg_sample.columns:
        alt_windows.append(win_col)
    else:
        print(f"NOTE: {win_col} not present in this panel -- re-run Steps 4/5/12 to include it.")

window_rows = []
for dv in alt_windows:
    if reg_sample[dv].isna().all():
        continue
    sub = reg_sample.dropna(subset=["Specificity", "Sentiment", dv])
    formula = f"Q('{dv}') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))"
    m = smf.ols(formula=formula, data=sub).fit(cov_type="HC3")
    for term in ["Specificity", "Sentiment"]:
        window_rows.append({
            "window": dv, "term": term, "coef": m.params[term], "se": m.bse[term],
            "p": m.pvalues[term], "N": int(m.nobs),
        })

table_windows = pd.DataFrame(window_rows)

# ------------------------------------------------------------
# Multiple-testing correction -- this table now runs len(alt_windows) x 2
# tests (one per window x Specificity/Sentiment). A single window's p<0.05
# in isolation is not enough grounds to call it a real effect at this
# test count; Holm (FWER) and Benjamini-Hochberg (FDR) both applied, same
# convention used elsewhere in this project (null_check/, esg_pillars/,
# esg_e_sg/).
# ------------------------------------------------------------
from statsmodels.stats.multitest import multipletests

reject_holm, p_holm, _, _ = multipletests(table_windows["p"], alpha=0.05, method="holm")
reject_bh, p_bh, _, _ = multipletests(table_windows["p"], alpha=0.05, method="fdr_bh")
table_windows["p_holm"] = p_holm
table_windows["sig_holm"] = reject_holm
table_windows["p_bh"] = p_bh
table_windows["sig_bh"] = reject_bh

print(f"Model 3, Firm+Year FE, HC3 SEs -- across CAR windows ({len(table_windows)} tests: "
      f"{len(alt_windows)} windows x Specificity/Sentiment)")
display(table_windows.round(4))

print(f"\nSurvive raw p<0.05: {int((table_windows['p'] < 0.05).sum())} of {len(table_windows)}")
print(f"Survive Holm-corrected p<0.05: {int(reject_holm.sum())} of {len(table_windows)}")
print(f"Survive Benjamini-Hochberg-corrected p<0.05: {int(reject_bh.sum())} of {len(table_windows)}")
if reject_holm.sum() > 0:
    print("\nHolm survivors:")
    display(table_windows.loc[reject_holm, ["window", "term", "coef", "p", "p_holm"]])
if reject_bh.sum() > 0:
    print("\nBH survivors:")
    display(table_windows.loc[reject_bh, ["window", "term", "coef", "p", "p_bh"]])


In [ ]:
# ============================================================
# Step 9.2 -- Influential observations: Cook's distance on the primary
# fit (Model 3, Spec C, CAR[-1,+1], HC3).
# ============================================================
from statsmodels.stats.outliers_influence import OLSInfluence

primary_fit = models_car_hc3[("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")]
influence = OLSInfluence(primary_fit)
cooks_d = influence.cooks_distance[0]

fit_sample = reg_sample.loc[primary_fit.model.data.row_labels].copy()
fit_sample["cooks_d"] = cooks_d
fit_sample["studentized_resid"] = influence.resid_studentized_external

threshold = 4 / primary_fit.nobs
flagged = fit_sample[fit_sample["cooks_d"] > threshold].sort_values("cooks_d", ascending=False)
print(f"Cook's distance threshold (4/N): {threshold:.4f}")
print(f"Flagged observations: {len(flagged)} of {int(primary_fit.nobs)}")
display(flagged[["Company", "Fiscal Year", "CAR[-1,+1]", "Specificity", "Sentiment",
                  "cooks_d", "studentized_resid"]])

aig_2019 = fit_sample[(fit_sample["Company"].str.contains("AIG")) & (fit_sample["Fiscal Year"] == 2019)]
covid_events = fit_sample[fit_sample["Fiscal Year"] == 2020]
print("\nAIG FY2019 specifically:")
if len(aig_2019):
    print(aig_2019[["Company", "Fiscal Year", "cooks_d", "studentized_resid"]].to_string(index=False))
else:
    print("not in complete-case sample")
print("\nAll FY2020 (COVID-period) events:")
if len(covid_events):
    print(covid_events[["Company", "Fiscal Year", "cooks_d", "studentized_resid"]].to_string(index=False))
else:
    print("none in complete-case sample")

# ------------------------------------------------------------
# Re-fit excluding whatever Cook's distance actually flagged (not a
# pre-assumed set) -- report whether Specificity/Sentiment survive.
# ------------------------------------------------------------
if len(flagged) > 0:
    excl_idx = flagged.index
    sub_excl = reg_sample.drop(index=[i for i in excl_idx if i in reg_sample.index])
    m_excl = smf.ols(
        formula="Q('CAR[-1,+1]') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))",
        data=sub_excl
    ).fit(cov_type="HC3")
    print(f"\nModel 3 / Spec C refit excluding {len(flagged)} flagged observation(s): "
          f"N={int(m_excl.nobs)}")
    for term in ["Specificity", "Sentiment"]:
        print(f"  {term}: coef={m_excl.params[term]:.4f}, se={m_excl.bse[term]:.4f}, "
              f"p={m_excl.pvalues[term]:.4f}")
else:
    print("\nNo observations exceeded the Cook's distance threshold -- no exclusion re-fit needed.")


In [ ]:
# ============================================================
# Step 9.3 -- Standard errors: HC3 (already default above), firm-
# clustered (already computed above), and a pairs-cluster bootstrap
# (resampling firms with replacement) as the feasible approximation to
# a full wild cluster bootstrap here.
#
# CAVEAT, stated explicitly rather than glossed over: with up to 22
# firms, NO clustering method -- analytic cluster-robust, wild
# bootstrap, or this pairs bootstrap -- has fully good asymptotic
# properties. The conventional guidance wants 30-50+ clusters. Reported
# because it was requested, and 22 clusters is closer to that guidance
# than the original 7-firm US-only pass, but still short of it.
# ============================================================
N_BOOTSTRAP = 2000
rng = np.random.default_rng(42)
firms = reg_sample["Company"].unique()

boot_coefs = {"Specificity": [], "Sentiment": []}
base_formula = "Q('CAR[-1,+1]') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))"

for b in range(N_BOOTSTRAP):
    sampled_firms = rng.choice(firms, size=len(firms), replace=True)
    # Re-key each resampled draw of a firm to its own dummy level, so
    # C(Company) doesn't silently merge two independent draws of the
    # same firm (which would happen if the same "Company" string
    # appeared twice with replace=True resampling).
    boot_frames = []
    for draw_i, f in enumerate(sampled_firms):
        block = reg_sample[reg_sample["Company"] == f].copy()
        block["Company"] = f"{f}__draw{draw_i}"
        boot_frames.append(block)
    boot_df = pd.concat(boot_frames, ignore_index=True)
    try:
        m_boot = smf.ols(formula=base_formula, data=boot_df).fit()
        for term in ["Specificity", "Sentiment"]:
            if term in m_boot.params.index:
                boot_coefs[term].append(m_boot.params[term])
    except Exception:
        continue

n_boot_success = len(boot_coefs["Specificity"])
print(f"Successful bootstrap draws: {n_boot_success} of {N_BOOTSTRAP} requested")

primary_model_key = ("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")
hc3_bse = models_car_hc3[primary_model_key].bse
cluster_bse = models_car_cluster[primary_model_key].bse
point_ests = models_car_hc3[primary_model_key].params

for term in ["Specificity", "Sentiment"]:
    vals = np.array(boot_coefs[term])
    if len(vals) > 10:
        se_boot = vals.std(ddof=1)
        ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
        print(f"\n{term}: point estimate={point_ests[term]:.4f}, "
              f"pairs-cluster-bootstrap SE={se_boot:.4f}, 95% pctile CI=[{ci_lo:.4f}, {ci_hi:.4f}]")
        print(f"  (compare to HC3 SE={hc3_bse[term]:.4f}, cluster SE={cluster_bse[term]:.4f})")
    else:
        print(f"\n{term}: too few successful bootstrap draws to report a stable SE/CI.")


## Step 9.4 — CAR[0,+30]/[0,+45]/[0,+60] follow-up: cluster/bootstrap SEs, and ESGSentiment substitution

Step 9.1's fuller window sweep (now 9 windows) shows Sentiment building up smoothly and peaking
around `CAR[0,+45]` (HC3 p=0.008), with `CAR[0,+30]` and `CAR[0,+60]` also significant
(p=0.018, p=0.041) -- Specificity stays null throughout. Three consecutive, monotonically-shaped
significant windows is a stronger signal than one isolated hit, but still needs: (a) the same
cluster/bootstrap rigor already applied to the primary CAR[-1,+1] result, and (b) a check of
whether ESGSentiment (not just general Sentiment) shows the same pattern. Run for all three
windows (`CAR[0,+30]`, `CAR[0,+45]`, `CAR[0,+60]`) rather than just the originally-flagged
`CAR[0,+60]`, since `CAR[0,+45]` is now the strongest point.

In [ ]:
# ============================================================
# Step 9.4.1 -- for each of CAR[0,+30]/[0,+45]/[0,+60], Model 3
# (Specificity + Sentiment), Firm+Year FE: HC3 vs. cluster(firm) vs.
# pairs-cluster bootstrap.
# ============================================================
followup_windows = ["CAR[0,+30]", "CAR[0,+45]", "CAR[0,+60]"]
followup_models = {}
followup_boot = {}

for win in followup_windows:
    formula = f"Q('{win}') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))"
    m_hc3 = smf.ols(formula=formula, data=reg_sample).fit(cov_type="HC3")
    m_cluster = smf.ols(formula=formula, data=reg_sample).fit(
        cov_type="cluster", cov_kwds={"groups": reg_sample["Company"]}
    )
    followup_models[win] = {"hc3": m_hc3, "cluster": m_cluster}

    boot_coefs = {"Specificity": [], "Sentiment": []}
    for b in range(N_BOOTSTRAP):
        sampled_firms = rng.choice(firms, size=len(firms), replace=True)
        boot_frames = []
        for draw_i, f in enumerate(sampled_firms):
            block = reg_sample[reg_sample["Company"] == f].copy()
            block["Company"] = f"{f}__draw{draw_i}"
            boot_frames.append(block)
        boot_df = pd.concat(boot_frames, ignore_index=True)
        try:
            m_boot = smf.ols(formula=formula, data=boot_df).fit()
            for term in ["Specificity", "Sentiment"]:
                if term in m_boot.params.index:
                    boot_coefs[term].append(m_boot.params[term])
        except Exception:
            continue
    followup_boot[win] = boot_coefs

    vals = np.array(boot_coefs["Sentiment"])
    print(f"{win}, Model 3, Firm+Year FE -- Sentiment term")
    print(f"  HC3:            coef={m_hc3.params['Sentiment']:.4f}, se={m_hc3.bse['Sentiment']:.4f}, "
          f"p={m_hc3.pvalues['Sentiment']:.4f}")
    print(f"  cluster(firm):  coef={m_cluster.params['Sentiment']:.4f}, "
          f"se={m_cluster.bse['Sentiment']:.4f}, p={m_cluster.pvalues['Sentiment']:.4f}")
    if len(vals) > 10:
        se_boot = vals.std(ddof=1)
        ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
        ci_excludes_zero = ci_lo > 0 or ci_hi < 0
        print(f"  bootstrap ({len(vals)} draws): SE={se_boot:.4f}, 95% CI=[{ci_lo:.4f}, {ci_hi:.4f}], "
              f"excludes zero={ci_excludes_zero}")
        print(f"  Survives p<0.05 under: HC3={m_hc3.pvalues['Sentiment'] < 0.05}, "
              f"cluster={m_cluster.pvalues['Sentiment'] < 0.05}, bootstrap CI excludes 0={ci_excludes_zero}")
    else:
        print("  Too few successful bootstrap draws to report a stable SE/CI.")
    print()


In [ ]:
# ============================================================
# Step 9.4.2 -- same three windows, ESGSentiment substituted for
# Sentiment: is this a general-tone effect or ESG-specific?
# ============================================================
for win in followup_windows:
    formula_esg = f"Q('{win}') ~ Specificity + ESGSentiment + C(Company) + C(Q('Fiscal Year'))"
    m_esg_hc3 = smf.ols(formula=formula_esg, data=reg_sample).fit(cov_type="HC3")
    m_esg_cluster = smf.ols(formula=formula_esg, data=reg_sample).fit(
        cov_type="cluster", cov_kwds={"groups": reg_sample["Company"]}
    )

    m_gen_hc3 = followup_models[win]["hc3"]
    m_gen_cluster = followup_models[win]["cluster"]

    print(f"{win}, Model 3 with ESGSentiment substituted for Sentiment, Firm+Year FE")
    print(f"  HC3:            coef={m_esg_hc3.params['ESGSentiment']:.4f}, "
          f"se={m_esg_hc3.bse['ESGSentiment']:.4f}, p={m_esg_hc3.pvalues['ESGSentiment']:.4f}")
    print(f"  cluster(firm):  coef={m_esg_cluster.params['ESGSentiment']:.4f}, "
          f"se={m_esg_cluster.bse['ESGSentiment']:.4f}, p={m_esg_cluster.pvalues['ESGSentiment']:.4f}")
    print(f"  Specificity (same model): HC3 coef={m_esg_hc3.params['Specificity']:.4f}, "
          f"p={m_esg_hc3.pvalues['Specificity']:.4f}; "
          f"cluster p={m_esg_cluster.pvalues['Specificity']:.4f}")
    print(f"  (for comparison, general Sentiment: HC3 p={m_gen_hc3.pvalues['Sentiment']:.4f}, "
          f"cluster p={m_gen_cluster.pvalues['Sentiment']:.4f})")

    esg_significant = m_esg_hc3.pvalues["ESGSentiment"] < 0.05 or m_esg_cluster.pvalues["ESGSentiment"] < 0.05
    if esg_significant:
        print("  ESGSentiment is ALSO significant here -- effect may not be tone-general-specific.")
    else:
        print("  ESGSentiment is NOT significant here, unlike overall Sentiment -- consistent with "
              "a general-tone effect, not an ESG-specific one.")
    print()


## Step 9.5 — Export the regression sample

All data underlying Steps 7-9: event date, CAR (all windows), calendar-mismatch diagnostics,
Specificity/ESGSentiment (Corporate+tier1+ restricted) and overall Sentiment, per firm-year. Two
files: the 209-row complete-case regression sample (`reg_sample`), and the full 240-row merged
panel (`analysis`, includes the 31 firm-years excluded from the regression for missing
Specificity/Sentiment, so the exclusions themselves are traceable too).

In [ ]:
# ============================================================
# Step 9.4 -- Export the full regression sample and the full merged
# (pre-complete-case) panel, for external review.
# ============================================================
export_cols = [
    "Company", "Region", "Fiscal Year", "Event Date (adjusted)",
    "Estimation N", "Est_Calendar_Mismatch_N", "Alpha", "Beta",
    "CAR[-1,+1]", "CAR[-1,+1]_Calendar_Gaps", "CAR[-2,+2]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+15]",
    "CAR[0,+20]", "CAR[0,+30]", "CAR[0,+45]", "CAR[0,+60]", "|CAR|[-1,+1]",
    "n_corporate_tier1plus", "Specificity", "ESGSentiment",
    "total_sentences", "Sentiment",
]
export_cols = [c for c in export_cols if c in analysis.columns]

reg_sample_export_path = "/content/drive/MyDrive/phd/Data/ceo_letter_regression_sample_209.csv"
full_merged_export_path = "/content/drive/MyDrive/phd/Data/ceo_letter_merged_panel_240.csv"

reg_sample[export_cols].sort_values(["Company", "Fiscal Year"]).to_csv(reg_sample_export_path, index=False)
analysis[export_cols].sort_values(["Company", "Fiscal Year"]).to_csv(full_merged_export_path, index=False)

print(f"Regression sample ({len(reg_sample)} rows) saved to: {reg_sample_export_path}")
print(f"Full merged panel ({len(analysis)} rows, includes the 31 excluded-from-regression "
      f"firm-years) saved to: {full_merged_export_path}")
print(f"\nColumns: {export_cols}")
display(reg_sample[export_cols].sort_values(["Company", "Fiscal Year"]).head())


## Report-only summary

Findings only -- no interpretation of causality, and this analysis is explicitly exploratory
given N<=240 across up to 22 firms. *The analysis examines whether CEO-letter narrative
characteristics are associated with abnormal stock returns around annual-report publication
events* -- not that CEO letters cause stock-market movements.

In [ ]:
print("=" * 70)
print("Merge diagnostics")
print("=" * 70)
print(f"Event-study panel: {len(event_panel)} firm-years")
n_dropped = len(event_panel) - len(reg_sample)
print(f"Complete-case regression sample: {len(reg_sample)} firm-years "
      f"({n_dropped} dropped for missing Specificity/ESGSentiment/Sentiment)")

section_rule = "=" * 70
print(f"\n{section_rule}\nModel 3 (Specificity + Sentiment), Firm+Year FE -- primary results\n{section_rule}")
for dv_label, models_hc3, models_cluster in [
    ("CAR[-1,+1]", models_car_hc3, models_car_cluster),
    ("|CAR|[-1,+1]", models_abscar_hc3, models_abscar_cluster),
]:
    m_hc3 = models_hc3[("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")]
    m_cl = models_cluster[("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")]
    print(f"\nDV = {dv_label}, N={int(m_hc3.nobs)}, R2={m_hc3.rsquared:.4f}")
    for term in ["Specificity", "Sentiment"]:
        print(f"  {term}: coef={m_hc3.params[term]:.4f}, "
              f"HC3 se={m_hc3.bse[term]:.4f} (p={m_hc3.pvalues[term]:.4f}), "
              f"cluster se={m_cl.bse[term]:.4f} (p={m_cl.pvalues[term]:.4f})")

n_firms_in_sample = reg_sample["Company"].nunique()
print(f"\n{section_rule}\nInterpretation (conservative, per instruction)\n{section_rule}")
print("The analysis examines whether CEO-letter narrative characteristics are associated with")
print("abnormal stock returns around annual-report publication events -- not that CEO letters")
print(f"cause stock-market movements. N={len(reg_sample)} firm-years across {n_firms_in_sample} "
      f"firms; firm-clustered and bootstrap SEs both carry the small-cluster caveat noted in "
      f"Step 9.3.")


---
# Part 3 — Non-US extension: ticker verification (15 companies)

Extends the event study beyond the 7 US firms to the other 15 companies in the full 22-firm
panel (Canada, China/HK, France, Germany, Italy, Switzerland, Japan). **This cell only verifies
candidate tickers resolve on Yahoo Finance -- it does not pull price data or run any event-study
computation.** Candidate tickers below are my best-guess identifiers for each company's primary
listing; none have been empirically confirmed yet. Run this cell, inspect the output, and report
back which resolve before any price pull is attempted -- same "probe before pulling" pattern used
for the ACE/Chubb ticker resolution in Part 1.

**Two companies have a name change in the source spreadsheet, analogous to the Chubb/ACE
situation, but NOT (as far as I can tell) a ticker change** -- both are the same legal entity
before and after a rename, still listed under the same Tokyo Stock Exchange code throughout:

- Dai-ichi: `'Dai-ichi Life Insurance Company / Dai-ichi Life Holdings'` (FY2012-2015) →
  `'Dai-ichi Life Holdings'` (FY2016-2023). Candidate ticker `8750.T` for both.
- Sompo: `'NKSJ Holdings / Sompo Holdings'` (FY2012-2013) → `'Sompo Holdings'` (FY2014-2023).
  Candidate ticker `8630.T` for both.

This is an assumption, not a confirmed fact -- the probe below checks coverage across the FULL
2012-2023 span for both tickers specifically to test it, the same way Part 1 empirically tested
(rather than assumed) whether `CB` covered the pre-2016 ACE years.

In [ ]:
# ============================================================
# Non-US candidate tickers -- 15 companies. UNVERIFIED candidates;
# this cell's job is to test them, not assume they are right.
# ============================================================
NONUS_COMPANY_TICKER = {
    # Canada
    "Power Corporation of Canada": "POW.TO",
    # China / Hong Kong
    "China Life Insurance Company Limited": "2628.HK",
    "People's Insurance Company of China (PICC)": "1339.HK",
    "Ping An Insurance Group": "2318.HK",
    # France
    "AXA SA": "CS.PA",
    # Germany
    "Allianz SE": "ALV.DE",
    "Munich Re": "MUV2.DE",
    "Talanx AG": "TLX.DE",
    # Italy
    "Assicurazioni Generali": "G.MI",
    # Switzerland
    "Swiss Re": "SREN.SW",
    "Zurich Insurance Group": "ZURN.SW",
    # Japan
    "Dai-ichi Life Holdings": "8750.T",
    "Dai-ichi Life Insurance Company / Dai-ichi Life Holdings": "8750.T",
    "MS&AD Insurance Group Holdings": "8725.T",
    "Sompo Holdings": "8630.T",
    "NKSJ Holdings / Sompo Holdings": "8630.T",
    "Tokio Marine Holdings": "8766.T",
}

# Distinct tickers to probe (the two Japan renames collapse to one ticker each).
NONUS_TICKERS = sorted(set(NONUS_COMPANY_TICKER.values()))
print(f"{len(NONUS_COMPANY_TICKER)} company-name entries -> {len(NONUS_TICKERS)} distinct tickers to probe")
for t in NONUS_TICKERS:
    companies = [c for c, tk in NONUS_COMPANY_TICKER.items() if tk == t]
    print(f"  {t}: {companies}")


In [ ]:
# ============================================================
# Probe each ticker in two windows: an EARLY window (2012, the panel's
# first fiscal year) and a RECENT window (2023), non-fatally. This
# mirrors the ACE/CB pattern (probe before pulling), but checks BOTH
# ends of the range up front, since for Dai-ichi/Sompo specifically we
# need to know whether one ticker really covers the full 2012-2023 span
# under the assumed no-ticker-change reading above -- not just whether
# it resolves at all today.
# ============================================================
import yfinance as yf

EARLY_PROBE = ("2012-01-01", "2012-02-15")
RECENT_PROBE = ("2023-01-01", "2023-02-15")

probe_results = {}
for ticker in NONUS_TICKERS:
    row = {"ticker": ticker}
    for label, window in [("early_2012", EARLY_PROBE), ("recent_2023", RECENT_PROBE)]:
        start, end = window
        try:
            data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
            row[label] = "OK" if len(data) > 0 else "EMPTY (0 rows, no error)"
        except Exception as e:
            row[label] = f"FAILED: {e}"
    probe_results[ticker] = row
    early_status = row["early_2012"]
    recent_status = row["recent_2023"]
    print(f"{ticker}: early_2012={early_status!r}, recent_2023={recent_status!r}")

probe_df = pd.DataFrame(probe_results).T
print()
display(probe_df)

# ------------------------------------------------------------
# Flag anything that needs a decision before pulling: a ticker that
# fails EITHER window (candidate is wrong / listing changed / delisted)
# or that resolves only in one window (possible real ticker change,
# same situation as ACE -> CB, needing the same kind of splice logic).
# ------------------------------------------------------------
problem_tickers = probe_df[
    probe_df["early_2012"].str.contains("FAILED|EMPTY")
    | probe_df["recent_2023"].str.contains("FAILED|EMPTY")
]
n_problem = len(problem_tickers)
n_total = len(probe_df)
print(f"\nTickers needing attention before any price pull: {n_problem} of {n_total}")
if n_problem:
    display(problem_tickers)
else:
    print("All candidate tickers resolved in both the 2012 and 2023 windows.")


In [ ]:
# ============================================================
# Benchmark check -- confirm URTH (already used for the US 7-firm
# panel) also covers back to 2012, since the non-US event windows will
# be compared against the same benchmark.
# ============================================================
try:
    urth_2012 = yf.download("URTH", start="2012-01-01", end="2012-02-15", progress=False, auto_adjust=False)
    status = "OK" if len(urth_2012) > 0 else "EMPTY"
    n_rows = len(urth_2012)
    print(f"URTH 2012 probe: {status} ({n_rows} rows)")
except Exception as e:
    print(f"URTH 2012 probe FAILED: {e}")


## Step 10 — Non-US event dates and canonical company names

Builds the 15-company non-US firm-year table from the same source file, and canonicalizes the
Dai-ichi and Sompo name-change pairs to a single `Company` label each (both are the same legal
entity, same ticker throughout -- confirmed in Part 3's probe, not assumed).

In [ ]:
# ============================================================
# Step 10.1 -- Non-US firm-year table + canonical company names
# ============================================================
df_nonus = df_dates_raw[df_dates_raw["Region"] != "United States"].copy()
df_nonus["Publication Date_raw"] = df_nonus["Publication Date"].astype(str)
# Some rows (following the user's manual re-verification of Japanese
# Integrated/Annual Report dates against primary sources, since the prior
# EDINET-derived dates for 5 Japanese companies were the wrong document --
# a regulatory securities-report filing deadline, not the actual investor
# Annual/Integrated Report publication date) are now NaN (day genuinely
# unverifiable) or a "YYYY-MM" month-only string (day not independently
# confirmed). Both are flagged, NOT dropped from df_nonus and NOT given an
# invented day -- Step 12 skips them explicitly rather than silently using
# a wrong or made-up event date.
df_nonus["Has_Day_Precision_Date"] = df_nonus["Publication Date_raw"].str.match(r"^\d{4}-\d{2}-\d{2}")
df_nonus["Publication Date"] = pd.to_datetime(df_nonus["Publication Date"], format="mixed", errors="coerce")
print(f"Non-US rows: {len(df_nonus)}")
n_no_day_precision = (~df_nonus["Has_Day_Precision_Date"]).sum()
print(f"Rows WITHOUT a verified exact publication day (NaN or month-only -- excluded from the "
      f"event-study loop in Step 12, not given an invented day): {n_no_day_precision} of {len(df_nonus)}")
if n_no_day_precision > 0:
    display(df_nonus.loc[~df_nonus["Has_Day_Precision_Date"],
                          ["Company", "Fiscal Year", "Publication Date_raw", "Note"]])
print(f"Distinct raw company-name strings: {df_nonus['Company'].nunique()}")

NONUS_CANONICAL_COMPANY = {
    "Power Corporation of Canada": "Power Corporation of Canada",
    "China Life Insurance Company Limited": "China Life Insurance Company Limited",
    "People\'s Insurance Company of China (PICC)": "People\'s Insurance Company of China (PICC)",
    "Ping An Insurance Group": "Ping An Insurance Group",
    "AXA SA": "AXA SA",
    "Allianz SE": "Allianz SE",
    "Munich Re": "Munich Re",
    "Talanx AG": "Talanx AG",
    "Assicurazioni Generali": "Assicurazioni Generali",
    "Swiss Re": "Swiss Re",
    "Zurich Insurance Group": "Zurich Insurance Group",
    "Dai-ichi Life Holdings": "Dai-ichi Life Holdings",
    "Dai-ichi Life Insurance Company / Dai-ichi Life Holdings": "Dai-ichi Life Holdings",
    "MS&AD Insurance Group Holdings": "MS&AD Insurance Group Holdings",
    "Sompo Holdings": "Sompo Holdings",
    "NKSJ Holdings / Sompo Holdings": "Sompo Holdings",
    "Tokio Marine Holdings": "Tokio Marine Holdings",
}

df_nonus["Company_canonical"] = df_nonus["Company"].map(NONUS_CANONICAL_COMPANY)
n_unmapped = df_nonus["Company_canonical"].isna().sum()
if n_unmapped > 0:
    print("UNMAPPED raw company names (STOP, fix NONUS_CANONICAL_COMPANY before continuing):")
    print(sorted(df_nonus.loc[df_nonus["Company_canonical"].isna(), "Company"].unique()))
assert n_unmapped == 0, f"{n_unmapped} non-US rows have no canonical company mapping -- STOP."

n_canonical_companies = df_nonus["Company_canonical"].nunique()
print(f"Canonical companies: {n_canonical_companies} (expected 15)")
assert n_canonical_companies == 15, f"Expected 15 canonical non-US companies, got {n_canonical_companies}"

counts_per_company = df_nonus.groupby("Company_canonical").size()
print("\nRows per canonical company (Dai-ichi and Sompo should each sum their two name-variant "
      "rows to 12):")
print(counts_per_company.to_string())
assert (counts_per_company == 12).all(), "Not every canonical company has exactly 12 fiscal years -- STOP."
print("\nCONFIRMED: 180 non-US firm-years across 15 canonical companies x 12 fiscal years.")


## Step 11 — Non-US price pull and return series

Reuses `NONUS_COMPANY_TICKER` from Part 3. No splice logic needed here (unlike Chubb/ACE) --
Part 3's probe confirmed a single ticker per canonical company covers the full 2012-2023 range,
except PICC (`1339.HK`) and Talanx (`TLX.DE`), whose actual price history starts partway through
2012 because both IPO'd that year (confirmed via their empty-early-2012 / OK-2023 probe results,
consistent with PICC's Dec-2012 HKEX listing and Talanx's Jun-2012 Frankfurt listing) -- not a
splice case, just a real, later start date that Step 12's existing
`MIN_ESTIMATION_OBS` check will handle by excluding their FY2012 (and possibly FY2013) event
naturally, the same way any other data-gap firm-year already gets excluded.

In [ ]:
# ============================================================
# Step 11.1 -- Pull daily price data for the 15 non-US tickers
# (URTH already pulled/available as `benchmark_returns` from Step 3-4).
# ============================================================
raw_prices_nonus = {}
for ticker in NONUS_TICKERS:
    print(f"Pulling {ticker}...")
    data = yf.download(ticker, start=PRICE_START, end=PRICE_END, progress=False, auto_adjust=True)
    raw_prices_nonus[ticker] = data
    if len(data) > 0:
        print(f"  {len(data)} rows, {data.index.min()} to {data.index.max()}")
    else:
        print("  NO DATA -- STOP, investigate before continuing (Part 3 probe said this resolved).")

raw_prices_nonus_dir = "/content/drive/MyDrive/phd/Data/event_study_raw_prices_nonus"
os.makedirs(raw_prices_nonus_dir, exist_ok=True)
for ticker, data in raw_prices_nonus.items():
    out_path = os.path.join(raw_prices_nonus_dir, f"{ticker}.csv")
    data.to_csv(out_path)
print(f"\nAll raw pulls saved to: {raw_prices_nonus_dir}")


In [ ]:
# ============================================================
# Step 11.2 -- Build one return series per canonical company. No
# splice needed (see markdown above) -- just each company\'s single
# ticker\'s own pct_change.
# ============================================================
def get_adj_close_from(price_dict, ticker):
    close = price_dict[ticker]["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.squeeze("columns")
    return close.dropna()

TICKER_BY_CANONICAL = {}
for raw_name, canonical in NONUS_CANONICAL_COMPANY.items():
    TICKER_BY_CANONICAL[canonical] = NONUS_COMPANY_TICKER[raw_name]

company_returns_nonus = {}
for canonical, ticker in TICKER_BY_CANONICAL.items():
    close = get_adj_close_from(raw_prices_nonus, ticker)
    returns = close.pct_change().dropna()
    returns.name = canonical
    company_returns_nonus[canonical] = returns
    print(f"{canonical} ({ticker}): {len(returns)} return observations, "
          f"{returns.index.min()} to {returns.index.max()}")


## Step 12 — Non-US market-model event study (calendar-mismatch aware)

Same market-model mechanics as Step 4 (estimation window `[-250,-30]`, event windows summed as
abnormal-return CAR, `MIN_ESTIMATION_OBS = 100`), but **each non-US firm trades on its own local
exchange calendar (Tokyo, Hong Kong, Frankfurt, Paris, Milan, Zurich, Toronto), while the
benchmark `URTH` trades on the US calendar** -- so a day the firm trades and the benchmark doesn't
(a local holiday that isn't a US holiday, or vice versa) is a real, expected mismatch, far more
frequent than for the 7 US companies (which share URTH's own calendar almost exactly). Step 4's
original code silently dropped these days from the abnormal-return sum via `.sum(skipna=True)` --
here that drop is counted and reported explicitly per firm-year, for both the estimation window
and the primary event window, rather than left invisible.

In [ ]:
# ============================================================
# Step 12.1 -- Event-study loop, non-US firms, with explicit
# calendar-mismatch counting/reporting.
# ============================================================
# A firm-year's estimation window is excluded if more than this fraction
# of its trading days have no matching benchmark (URTH) return. Ordinary
# country-holiday mismatches run ~2-4% across this panel; Dai-ichi
# Life Holdings FY2012 hit ~33% because its window reaches back before
# URTH itself started trading (2012-01-17) -- a data-availability gap,
# not routine holiday noise. 0.15 separates that case cleanly from
# everything else observed without touching any other firm-year.
CALENDAR_MISMATCH_THRESHOLD = 0.15

results_nonus = []
n_shifted_nonus = 0
excluded_thin_estimation_nonus = []
excluded_no_day_precision_nonus = []
excluded_calendar_mismatch_nonus = []

for _, row in df_nonus.iterrows():
    company = row["Company_canonical"]
    fiscal_year = row["Fiscal Year"]
    raw_event_date = row["Publication Date"]

    if not row["Has_Day_Precision_Date"]:
        excluded_no_day_precision_nonus.append((company, fiscal_year, row["Publication Date_raw"]))
        continue

    firm_ret = company_returns_nonus[company]
    firm_calendar = firm_ret.index

    if raw_event_date in firm_calendar:
        event_date = raw_event_date
    else:
        event_date = next_trading_day(raw_event_date, firm_calendar)
        n_shifted_nonus += 1
        if event_date is None:
            print(f"SKIPPING {company} FY{fiscal_year}: no trading day found on/after "
                  f"{raw_event_date} in the available price history.")
            continue

    # ---- estimation window ----
    est_start = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[0])
    est_end = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[1])
    if est_start is None or est_end is None:
        print(f"SKIPPING {company} FY{fiscal_year}: estimation window falls outside available "
              f"price history.")
        continue

    est_mask = (firm_ret.index >= est_start) & (firm_ret.index <= est_end)
    est_firm = firm_ret[est_mask]
    est_bench_raw = benchmark_returns.reindex(est_firm.index)
    est_calendar_mismatch_n = int(est_bench_raw.isna().sum())
    total_est_window_days = len(est_firm)
    est_mismatch_fraction = est_calendar_mismatch_n / total_est_window_days if total_est_window_days > 0 else 0.0
    if est_mismatch_fraction > CALENDAR_MISMATCH_THRESHOLD:
        excluded_calendar_mismatch_nonus.append(
            (company, fiscal_year, est_calendar_mismatch_n, total_est_window_days, est_mismatch_fraction)
        )
        continue
    est_bench = est_bench_raw.dropna()
    est_common = est_firm.index.intersection(est_bench.index)
    est_firm = est_firm.loc[est_common]
    est_bench = benchmark_returns.loc[est_common]

    n_est = len(est_common)
    if n_est < MIN_ESTIMATION_OBS:
        excluded_thin_estimation_nonus.append((company, fiscal_year, n_est))
        continue

    X = np.column_stack([np.ones(n_est), est_bench.values])
    coefs, _, _, _ = np.linalg.lstsq(X, est_firm.values, rcond=None)
    alpha, beta = coefs[0], coefs[1]

    # ---- event windows, with calendar-mismatch counted per window ----
    car_values = {}
    car_calendar_gaps = {}
    for win_name, (lo, hi) in EVENT_WINDOWS.items():
        win_start = trading_day_offset(event_date, firm_calendar, lo)
        win_end = trading_day_offset(event_date, firm_calendar, hi)
        if win_start is None or win_end is None:
            car_values[win_name] = np.nan
            car_calendar_gaps[win_name] = np.nan
            continue
        win_mask = (firm_ret.index >= win_start) & (firm_ret.index <= win_end)
        win_firm = firm_ret[win_mask]
        win_bench = benchmark_returns.reindex(win_firm.index)
        gap_n = int(win_bench.isna().sum())
        car_calendar_gaps[win_name] = gap_n
        abnormal = win_firm - (alpha + beta * win_bench)
        car_values[win_name] = abnormal.sum()  # skipna=True by default -- gap_n above makes this explicit

    results_nonus.append({
        "Company": company,
        "Fiscal Year": fiscal_year,
        "Event Date (raw)": raw_event_date,
        "Event Date (adjusted)": event_date,
        "Shifted": event_date != raw_event_date,
        "Estimation N": n_est,
        "Est_Calendar_Mismatch_N": est_calendar_mismatch_n,
        "Alpha": alpha,
        "Beta": beta,
        "CAR[-1,+1]": car_values["primary_-1_+1"],
        "CAR[-1,+1]_Calendar_Gaps": car_calendar_gaps["primary_-1_+1"],
        "CAR[-2,+2]": car_values["robustness_-2_+2"],
        "CAR[0,+1]": car_values["robustness_0_+1"],
        "CAR[0,+5]": car_values["robustness_0_+5"],
        "CAR[0,+15]": car_values["robustness_0_+15"],
        "CAR[0,+20]": car_values["robustness_0_+20"],
        "CAR[0,+30]": car_values["robustness_0_+30"],
        "CAR[0,+45]": car_values["robustness_0_+45"],
        "CAR[0,+60]": car_values["robustness_0_+60"],
        "|CAR|[-1,+1]": abs(car_values["primary_-1_+1"]),
    })

print(f"Publication dates shifted to next trading day: {n_shifted_nonus} of {len(df_nonus)}")
print(f"\nFirm-years excluded for <{MIN_ESTIMATION_OBS} estimation-window observations: "
      f"{len(excluded_thin_estimation_nonus)}")
for company, fy, n in excluded_thin_estimation_nonus:
    print(f"  {company} FY{fy}: only {n} estimation-window observations")

print(f"\nFirm-years excluded for no verified exact publication day (NaN or month-only in the "
      f"source spreadsheet -- see Step 10.1): {len(excluded_no_day_precision_nonus)}")
for company, fy, raw_val in excluded_no_day_precision_nonus:
    print(f"  {company} FY{fy}: Publication Date = {raw_val!r}")

print(f"\nFirm-years excluded for estimation-window calendar-mismatch fraction > "
      f"{CALENDAR_MISMATCH_THRESHOLD:.0%} (benchmark data gap, not routine holiday noise): "
      f"{len(excluded_calendar_mismatch_nonus)}")
for company, fy, mismatch_n, total_n, frac in excluded_calendar_mismatch_nonus:
    print(f"  {company} FY{fy}: {mismatch_n} of {total_n} estimation-window days ({frac:.1%}) "
          f"had no matching benchmark return")


In [ ]:
# ============================================================
# Step 12.2 -- Calendar-mismatch report. Explicit, not folded silently
# into the CAR sums above.
# ============================================================
panel_nonus = pd.DataFrame(results_nonus)
panel_nonus = panel_nonus.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

print(f"Firm-years in non-US panel: {len(panel_nonus)} of {len(df_nonus)} original non-US firm-years")
print(f"({len(df_nonus) - len(panel_nonus)} excluded -- see Step 12.1\'s exclusion list above for why)")

est_mismatch = panel_nonus[panel_nonus["Est_Calendar_Mismatch_N"] > 0]
print(f"\nFirm-years with >=1 estimation-window calendar-mismatch day "
      f"(firm trades, URTH doesn\'t, or vice versa): {len(est_mismatch)} of {len(panel_nonus)}")
if len(est_mismatch):
    display(est_mismatch[["Company", "Fiscal Year", "Estimation N", "Est_Calendar_Mismatch_N"]]
            .sort_values("Est_Calendar_Mismatch_N", ascending=False))

primary_mismatch = panel_nonus[panel_nonus["CAR[-1,+1]_Calendar_Gaps"] > 0]
print(f"\nFirm-years with >=1 calendar-mismatch day INSIDE the primary CAR[-1,+1] window "
      f"(only 3 trading days wide -- a single dropped day is a meaningfully larger share of this "
      f"window than of the ~220-day estimation window): {len(primary_mismatch)} of {len(panel_nonus)}")
if len(primary_mismatch):
    display(primary_mismatch[["Company", "Fiscal Year", "CAR[-1,+1]", "CAR[-1,+1]_Calendar_Gaps"]])
else:
    print("None -- every non-US firm-year\'s primary event window had a matching URTH trading day "
          "for all 3 days.")


## Step 13 — Combined 22-company panel

In [ ]:
# ============================================================
# Step 13.1 -- Combine the US (Part 1) and non-US (Part 3) panels into
# one 22-company panel. Non-US-only diagnostic columns
# (Est_Calendar_Mismatch_N, CAR[-1,+1]_Calendar_Gaps) are kept, filled
# with 0 for the US rows (genuinely near-zero mismatch expected there,
# not imputed as unknown).
# ============================================================
panel_us_tagged = panel[display_cols].copy()
panel_us_tagged["Region"] = "United States"
panel_us_tagged["Est_Calendar_Mismatch_N"] = 0
panel_us_tagged["CAR[-1,+1]_Calendar_Gaps"] = 0

region_by_canonical = (
    df_nonus.drop_duplicates("Company_canonical")
    .set_index("Company_canonical")["Region"]
    .to_dict()
)
panel_nonus_tagged = panel_nonus.copy()
panel_nonus_tagged["Region"] = panel_nonus_tagged["Company"].map(region_by_canonical)

combined_cols = ["Company", "Region", "Fiscal Year", "Event Date (adjusted)", "Estimation N",
                  "Est_Calendar_Mismatch_N", "Alpha", "Beta", "CAR[-1,+1]", "CAR[-1,+1]_Calendar_Gaps",
                  "CAR[-2,+2]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+15]", "CAR[0,+20]", "CAR[0,+30]",
                  "CAR[0,+45]", "CAR[0,+60]", "|CAR|[-1,+1]"]
panel_full = pd.concat([panel_us_tagged, panel_nonus_tagged], ignore_index=True, sort=False)
panel_full = panel_full[combined_cols].sort_values(["Region", "Company", "Fiscal Year"]).reset_index(drop=True)

n_companies_full = panel_full["Company"].nunique()
print(f"Combined panel: {len(panel_full)} firm-years ({n_companies_full} companies)")
print(panel_full.groupby("Region").size().to_string())

output_path_nonus = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_nonUS.csv"
panel_nonus_tagged[combined_cols[:1] + combined_cols[2:]].to_csv(output_path_nonus, index=False)
print(f"\nSaved non-US-only panel to: {output_path_nonus}")

output_path_full = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_full22.csv"
panel_full.to_csv(output_path_full, index=False)
print(f"Saved combined 22-company panel to: {output_path_full}")

display(panel_full)


---
# Part 4 — Credible optimism vs. cheap-talk: 3-D ESG cluster interaction with CAR

New hypothesis, building on the 3-D KMeans clustering from the specificity-pipeline notebook
(`PHDp2_CEOLetters_AnnualReports_TextAnalysis.ipynb`, Section 8e) and confirmed centroid
interpretation from the valuation-regression notebook (`PHDp2_FinancialData_Regression.ipynb`,
cell 102): **cluster 0** = high-volume/low-specificity ("cheap-talk" optimism), **cluster 1** =
high-specificity ("credible" optimism), **cluster 2** = minimal ESG disclosure. Tests whether the
market reacts differently to credible-optimism firm-years than to cheap-talk firm-years, using
CAR rather than Tobin's Q as the outcome.

Entirely new cells appended after Step 13 -- Steps 1-13 above are untouched. Uses three sources
not otherwise loaded in this notebook: `df_ar_cluster.csv` (cluster assignment), plus
`Combined_Financials_With_ROA.csv` and `TobinsQ_Results_FullNames.csv` (financial controls),
matching the same merge the valuation-regression notebook performs.

**Step 14.1 is diagnostic-only** -- it loads all three sources and prints every company-name
spelling actually present, without guessing a mapping. The financial data uses a THIRD naming
convention, different from both this notebook's `Company` column and the short `df_ar_*` names
(confirmed by reading the source notebook's own `name_fixes` dict, not assumed) -- guessing a
merge mapping here risks a silent 0-row mismatch for any misspelled company, so the actual
mapping is built only after this diagnostic's output is reviewed.

In [ ]:
# ============================================================
# Step 14.1 -- Load cluster + financial data, print all company-name
# spellings. Diagnostic only -- no merge yet.
# ============================================================
cluster_path = "/content/drive/MyDrive/phd/Data/df_ar_cluster.csv"
financials_path = "/content/drive/MyDrive/phd/BloombergData/Combined_Financials_With_ROA.csv"
tobinsq_path = "/content/drive/MyDrive/phd/Data/TobinsQ_Results_FullNames.csv"

df_cluster_3d = pd.read_csv(cluster_path)
print(f"df_ar_cluster.csv: {df_cluster_3d.shape}")
print("Columns:", df_cluster_3d.columns.tolist())

df_fin_raw = pd.read_csv(financials_path)
print(f"\nCombined_Financials_With_ROA.csv: {df_fin_raw.shape}")
print("Columns:", df_fin_raw.columns.tolist())

df_tobinsq_raw = pd.read_csv(tobinsq_path)
print(f"\nTobinsQ_Results_FullNames.csv: {df_tobinsq_raw.shape}")
print("Columns:", df_tobinsq_raw.columns.tolist())

# ------------------------------------------------------------
# Every company-name spelling actually present, on all relevant sides,
# printed in full rather than inferred from the other notebook's
# name_fixes dict (that dict maps the SHORT df_ar_ names to the
# financial data's names -- it does not by itself tell us what this
# notebook's own long-form Company values need to map to).
# ------------------------------------------------------------
print(f"\ndf_ar_cluster.csv 'Company Name' values ({df_cluster_3d['Company Name'].nunique()}): "
      f"{sorted(df_cluster_3d['Company Name'].unique())}")
print(f"\nCombined_Financials_With_ROA.csv 'Company' values "
      f"({df_fin_raw['Company'].nunique()}): {sorted(df_fin_raw['Company'].unique())}")
print(f"\nTobinsQ_Results_FullNames.csv 'Company' values "
      f"({df_tobinsq_raw['Company'].nunique()}): {sorted(df_tobinsq_raw['Company'].unique())}")

if "Region" in df_fin_raw.columns:
    print(f"\nCombined_Financials_With_ROA.csv 'Region' values: {sorted(df_fin_raw['Region'].unique())}")
if "Region" in df_tobinsq_raw.columns:
    print(f"TobinsQ_Results_FullNames.csv 'Region' values: {sorted(df_tobinsq_raw['Region'].unique())}")

print(f"\nThis notebook's own 22 panel companies (from Step 6.1's SHORT_TO_LONG_NAME values): "
      f"{sorted(set(SHORT_TO_LONG_NAME.values()))}")

# Merge financials + TobinsQ exactly as the valuation-regression
# notebook does, so the diagnostic reflects the actual post-merge
# company/year/region coverage, not just each file in isolation.
df_financials_full = pd.merge(
    df_fin_raw, df_tobinsq_raw, on=["Company", "Year", "Region"], how="inner"
)
print(f"\ndf_financials_full (post Company/Year/Region merge): {df_financials_full.shape}")
print("Columns:", df_financials_full.columns.tolist())
for col in ["Leverage", "Size", "ROA", "Revenue Growth YoY (%)"]:
    print(f"  {col!r} present: {col in df_financials_full.columns}")


## Step 14.2 — Build the verified merge: cluster assignment + financial controls into the panel

Confirmed directly from Step 14.1's output (not guessed):
- `df_ar_cluster.csv`'s 'Company Name' spellings are IDENTICAL to the short names already covered
  by `SHORT_TO_LONG_NAME` (built and verified in Step 6.1) -- reused as-is, no new dict needed.
- The financial data (`Combined_Financials_With_ROA.csv` / `TobinsQ_Results_FullNames.csv`, merged
  on Company/Year/Region) uses a THIRD naming convention. `FIN_TO_EVENT_NAME` below is built from
  the exact 23/24 names Step 14.1 printed and verified to map all 22 panel companies with zero
  unmapped/leftover entries; 'New China Life' and 'India Life Insurance' are real companies in the
  broader Bloomberg dataset that are simply not part of this 22-firm panel -- expected, not an error.

**Design choice, stated explicitly rather than left implicit**: cluster assignment and financial
controls are merged **contemporaneously** (same Company + Fiscal Year as the CAR event), not
lagged. This differs from the valuation-regression notebook's `clag_*` convention (which lags
cluster because it's testing whether prior-year disclosure predicts current Tobin's Q) -- here the
CAR event window follows directly from THAT fiscal year's own letter, so the contemporaneous
match is the natural design for this specific hypothesis, not an oversight.

In [ ]:
# ============================================================
# Step 14.2 -- Merge cluster (via SHORT_TO_LONG_NAME) and financial
# controls (via FIN_TO_EVENT_NAME) into the 240-firm-year panel.
# ============================================================
FIN_TO_EVENT_NAME = {
    "AIG": "American International Group (AIG)",
    "AXA": "AXA SA",
    "Allianz": "Allianz SE",
    "Allstate": "The Allstate Corporation",
    "Assicurazioni Generali": "Assicurazioni Generali",
    "China Life": "China Life Insurance Company Limited",
    "Chubb": "Chubb",
    "Daichi": "Dai-ichi Life Holdings",
    "MSAD": "MS&AD Insurance Group Holdings",
    "MetLife": "MetLife, Inc.",
    "Munich RE": "Munich Re",
    "People Insurance China": "People's Insurance Company of China (PICC)",
    "Ping An": "Ping An Insurance Group",
    "Power Corp Canada": "Power Corporation of Canada",
    "Progressive": "The Progressive Corporation",
    "Prudentials": "Prudential Financial, Inc.",
    "Sompo": "Sompo Holdings",
    "Swiss RE": "Swiss Re",
    "Talanx": "Talanx AG",
    "Tokio Marine": "Tokio Marine Holdings",
    "Travelers": "The Travelers Companies, Inc.",
    "Zurich": "Zurich Insurance Group",
    # Present in the source files but not part of this 22-firm panel --
    # mapped to None deliberately so they show up as "unmapped" rather
    # than silently vanishing, and are excluded below by construction
    # (panel_companies_set filter), not by this mapping alone.
    "New China Life": None,
    "India Life Insurance": None,
}

panel_companies_set = set(SHORT_TO_LONG_NAME.values())

# ---- Cluster data ----
df_cluster_3d["Company"] = df_cluster_3d["Company Name"].map(SHORT_TO_LONG_NAME)
df_cluster_3d = df_cluster_3d.rename(columns={"Year": "Fiscal Year"})
unmapped_cluster = df_cluster_3d[df_cluster_3d["Company"].isna()]["Company Name"].unique()
print(f"Unmapped cluster company names: {sorted(unmapped_cluster)} (should be empty)")
assert len(unmapped_cluster) == 0, "Unmapped cluster company names -- STOP."

# ---- Financial controls ----
df_financials_full["Company_event"] = df_financials_full["Company"].map(FIN_TO_EVENT_NAME)
df_fin_panel = df_financials_full[df_financials_full["Company_event"].isin(panel_companies_set)].copy()

# Select only the needed columns (Company_event, not the original
# financial-data-spelled "Company") before renaming -- selecting first
# avoids ever having two columns named "Company" at once, which is what
# caused df_fin_panel["Company"] to return a DataFrame instead of a
# Series (AttributeError on .unique()) in an earlier version of this cell.
df_fin_panel = df_fin_panel[[
    "Company_event", "Year", "Leverage", "Size", "ROA", "Revenue Growth YoY (%)"
]].rename(columns={
    "Company_event": "Company",
    "Year": "Fiscal Year",
    "Revenue Growth YoY (%)": "Revenue_Growth",
})

mapped_fin_companies = set(df_fin_panel["Company"].unique())
print(f"\nPanel companies with NO rows at all in financial data after mapping: "
      f"{sorted(panel_companies_set - mapped_fin_companies)} (should be empty)")

# ---- Merge into the 240-firm-year panel (analysis, from Step 6.4) ----
panel14 = analysis.merge(
    df_cluster_3d[["Company", "Fiscal Year", "cluster_3d"]], on=["Company", "Fiscal Year"], how="left"
)
panel14 = panel14.merge(df_fin_panel, on=["Company", "Fiscal Year"], how="left")

print(f"\nPanel rows: {len(panel14)} (should still be {len(analysis)} -- left merges)")
assert len(panel14) == len(analysis), "Row count changed after merge -- STOP, check for duplicate keys."

n_missing_cluster = panel14["cluster_3d"].isna().sum()
n_missing_fin = panel14[["Leverage", "Size", "ROA", "Revenue_Growth"]].isna().any(axis=1).sum()
print(f"Missing cluster_3d: {n_missing_cluster} of {len(panel14)}")
print(f"Missing at least one financial control: {n_missing_fin} of {len(panel14)}")


## Step 14.3 — Build the two cluster dummies; report cell sizes before running anything

`Cluster1_HighSpec` = 1 iff `cluster_3d == 1` (high-specificity, "credible optimism").
`Cluster0_HighVolLowSpec` = 1 iff `cluster_3d == 0` (high-volume/low-specificity, "cheap-talk").
`cluster_3d == 2` (minimal) is the omitted reference category for both.

In [ ]:
# ============================================================
# Step 14.3 -- Cluster dummies + cell-size report (before any regression)
# ============================================================
panel14["Cluster1_HighSpec"] = (panel14["cluster_3d"] == 1).astype("Int64")
panel14["Cluster0_HighVolLowSpec"] = (panel14["cluster_3d"] == 0).astype("Int64")
panel14.loc[panel14["cluster_3d"].isna(), ["Cluster1_HighSpec", "Cluster0_HighVolLowSpec"]] = pd.NA

step14_controls = ["Leverage", "Size", "ROA", "Revenue_Growth"]
step14_complete = panel14.dropna(subset=["cluster_3d"] + step14_controls).copy()

print(f"Complete-case sample (cluster + all financial controls non-missing): "
      f"{len(step14_complete)} of {len(panel14)}")

cell_sizes = step14_complete["cluster_3d"].value_counts().sort_index()
cell_sizes.index = cell_sizes.index.map({0: "cluster 0 (high-vol/low-spec)",
                                          1: "cluster 1 (high-spec)",
                                          2: "cluster 2 (minimal, reference)"})
print("\nCell sizes (complete-case, before any per-window CAR restriction):")
print(cell_sizes.to_string())

MIN_CELL_SIZE = 15
thin_cells = cell_sizes[cell_sizes < MIN_CELL_SIZE]
if len(thin_cells) > 0:
    print(f"\nFLAG: {len(thin_cells)} cluster(s) below {MIN_CELL_SIZE} firm-years -- "
          f"coefficients for these groups should be read as exploratory, not precision estimates:")
    print(thin_cells.to_string())
else:
    print(f"\nAll clusters have at least {MIN_CELL_SIZE} firm-years in the complete-case sample.")


## Step 14.4 — Regression: CAR ~ cluster dummies + controls + firm FE + year FE

Four windows: `CAR[-1,+1]` (primary), `CAR[0,+30]`, `CAR[0,+45]`, `CAR[0,+60]` (the three
windows where general Sentiment showed something pre-correction in Step 9.1, kept here for
direct comparability). HC3 and firm-clustered SEs, each on its own complete-case sample.

In [ ]:
# ============================================================
# Step 14.4 -- Fit CAR ~ Cluster1_HighSpec + Cluster0_HighVolLowSpec +
# Controls + C(Company) + C(Fiscal Year), per window, HC3 and cluster SEs.
# ============================================================
step14_windows = ["CAR[-1,+1]", "CAR[0,+30]", "CAR[0,+45]", "CAR[0,+60]"]
step14_formula_rhs = (
    "Cluster1_HighSpec + Cluster0_HighVolLowSpec + Leverage + Size + ROA + Revenue_Growth "
    "+ C(Company) + C(Q('Fiscal Year'))"
)

step14_models = {}
step14_rows = []

for win in step14_windows:
    required = ["cluster_3d", "Cluster1_HighSpec", "Cluster0_HighVolLowSpec", win] + step14_controls
    sub = panel14.dropna(subset=required).copy()
    sub["Cluster1_HighSpec"] = sub["Cluster1_HighSpec"].astype(int)
    sub["Cluster0_HighVolLowSpec"] = sub["Cluster0_HighVolLowSpec"].astype(int)

    formula = f"Q('{win}') ~ {step14_formula_rhs}"
    m_hc3 = smf.ols(formula=formula, data=sub).fit(cov_type="HC3")
    m_cluster = smf.ols(formula=formula, data=sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub["Company"]}
    )
    step14_models[win] = {"hc3": m_hc3, "cluster": m_cluster, "sample": sub}

    win_cell_sizes = sub["cluster_3d"].value_counts().sort_index()
    print(f"{win}: N={len(sub)}, cell sizes = {win_cell_sizes.to_dict()}")

    for term in ["Cluster1_HighSpec", "Cluster0_HighVolLowSpec"]:
        step14_rows.append({
            "window": win, "term": term,
            "coef": m_hc3.params[term], "hc3_se": m_hc3.bse[term], "hc3_p": m_hc3.pvalues[term],
            "cluster_se": m_cluster.bse[term], "cluster_p": m_cluster.pvalues[term],
            "N": int(m_hc3.nobs),
        })

table_14_4 = pd.DataFrame(step14_rows)
print("\nTable 14.4 -- CAR ~ cluster dummies + controls + firm/year FE, both dummy coefficients")
display(table_14_4.round(4))


## Step 14.5 — The key test: pairwise Cluster1_HighSpec vs. Cluster0_HighVolLowSpec

Uses `model.t_test("Cluster1_HighSpec - Cluster0_HighVolLowSpec = 0")`, which draws on the
fitted model's actual coefficient covariance matrix (`cov_params()`) -- not an independence
assumption between the two dummy coefficients' standard errors. This is the number that answers
"does the market react differently to credible-optimism firms than to cheap-talk firms," as
distinct from each dummy's own comparison against the minimal-disclosure reference group.

In [ ]:
# ============================================================
# Step 14.5 -- Pairwise Cluster1_HighSpec vs. Cluster0_HighVolLowSpec,
# via the fitted model's own covariance matrix (t_test), both SE types.
# ============================================================
pairwise_rows = []
for win in step14_windows:
    m_hc3 = step14_models[win]["hc3"]
    m_cluster = step14_models[win]["cluster"]

    contrast = "Cluster1_HighSpec - Cluster0_HighVolLowSpec = 0"
    tt_hc3 = m_hc3.t_test(contrast)
    tt_cluster = m_cluster.t_test(contrast)

    # np.ravel(...)[0] instead of float(x[0]) -- statsmodels' t_test()
    # attributes are already length-1 arrays here, and indexing then
    # converting with float() on some of them raises a NumPy
    # DeprecationWarning (ndim>0 array to scalar); ravel()[0] is the
    # version-safe way to pull out the single value either way.
    pairwise_rows.append({
        "window": win,
        "diff (credible - cheap_talk)": float(np.ravel(tt_hc3.effect)[0]),
        "hc3_se": float(np.ravel(tt_hc3.sd)[0]), "hc3_p": float(np.ravel(tt_hc3.pvalue)[0]),
        "cluster_se": float(np.ravel(tt_cluster.sd)[0]), "cluster_p": float(np.ravel(tt_cluster.pvalue)[0]),
        "N": int(m_hc3.nobs),
    })

table_14_5 = pd.DataFrame(pairwise_rows)
print("Table 14.5 -- Pairwise test: credible optimism (Cluster1) vs. cheap-talk (Cluster0)")
display(table_14_5.round(4))


## Step 14.6 — Multiple-testing correction (Holm + Benjamini-Hochberg)

Two separate families, each corrected independently, same convention as Step 9.1:
1. The **individual dummy coefficients** vs. the minimal-disclosure reference (Table 14.4) --
   8 tests minimum (2 dummies x 4 windows), run separately for HC3 and cluster-robust p-values.
2. The **pairwise credible-vs-cheap-talk comparisons** (Table 14.5) -- 4 tests (1 per window),
   again run separately for HC3 and cluster-robust p-values.

Report-only: states plainly what survives each correction, not led with raw p-values.

In [ ]:
# ============================================================
# Step 14.6 -- Holm + BH correction, both families, both SE types.
# ============================================================
from statsmodels.stats.multitest import multipletests

def add_corrections(df, p_col, prefix):
    reject_holm, p_holm, _, _ = multipletests(df[p_col], alpha=0.05, method="holm")
    reject_bh, p_bh, _, _ = multipletests(df[p_col], alpha=0.05, method="fdr_bh")
    df[f"{prefix}_p_holm"] = p_holm
    df[f"{prefix}_sig_holm"] = reject_holm
    df[f"{prefix}_p_bh"] = p_bh
    df[f"{prefix}_sig_bh"] = reject_bh
    return df

table_14_4 = add_corrections(table_14_4, "hc3_p", "hc3")
table_14_4 = add_corrections(table_14_4, "cluster_p", "cluster")
table_14_5 = add_corrections(table_14_5, "hc3_p", "hc3")
table_14_5 = add_corrections(table_14_5, "cluster_p", "cluster")

print("=" * 70)
print("FAMILY 1 -- individual dummy coefficients vs. minimal-disclosure reference "
      f"({len(table_14_4)} tests per SE type)")
print("=" * 70)
display(table_14_4.round(4))
n_raw_1_hc3 = int((table_14_4["hc3_p"] < 0.05).sum())
n_holm_1_hc3 = int(table_14_4["hc3_sig_holm"].sum())
n_bh_1_hc3 = int(table_14_4["hc3_sig_bh"].sum())
n_raw_1_cl = int((table_14_4["cluster_p"] < 0.05).sum())
n_holm_1_cl = int(table_14_4["cluster_sig_holm"].sum())
n_bh_1_cl = int(table_14_4["cluster_sig_bh"].sum())
print(f"\nHC3: raw p<0.05 = {n_raw_1_hc3} of {len(table_14_4)}, Holm survives = {n_holm_1_hc3}, "
      f"BH survives = {n_bh_1_hc3}")
print(f"cluster(firm): raw p<0.05 = {n_raw_1_cl} of {len(table_14_4)}, Holm survives = {n_holm_1_cl}, "
      f"BH survives = {n_bh_1_cl}")

print(f"\n{'=' * 70}")
print(f"FAMILY 2 -- pairwise credible-vs-cheap-talk comparisons ({len(table_14_5)} tests per SE type)")
print("=" * 70)
display(table_14_5.round(4))
n_raw_2_hc3 = int((table_14_5["hc3_p"] < 0.05).sum())
n_holm_2_hc3 = int(table_14_5["hc3_sig_holm"].sum())
n_bh_2_hc3 = int(table_14_5["hc3_sig_bh"].sum())
n_raw_2_cl = int((table_14_5["cluster_p"] < 0.05).sum())
n_holm_2_cl = int(table_14_5["cluster_sig_holm"].sum())
n_bh_2_cl = int(table_14_5["cluster_sig_bh"].sum())
print(f"\nHC3: raw p<0.05 = {n_raw_2_hc3} of {len(table_14_5)}, Holm survives = {n_holm_2_hc3}, "
      f"BH survives = {n_bh_2_hc3}")
print(f"cluster(firm): raw p<0.05 = {n_raw_2_cl} of {len(table_14_5)}, Holm survives = {n_holm_2_cl}, "
      f"BH survives = {n_bh_2_cl}")

print(f"\n{'=' * 70}\nBOTTOM LINE\n{'=' * 70}")
any_survive = (n_holm_1_hc3 + n_holm_1_cl + n_bh_1_hc3 + n_bh_1_cl
               + n_holm_2_hc3 + n_holm_2_cl + n_bh_2_hc3 + n_bh_2_cl) > 0
if any_survive:
    print("At least one test survives correction -- see the tables above for which specific "
          "window/term/SE-type combination(s).")
else:
    print("Nothing survives either Holm or Benjamini-Hochberg correction in either family, under "
          "either SE type. Any raw p<0.05 hits above should not be reported as significant "
          "findings on their own.")


---
# Part 5 — Did NFRD change the Specificity/Sentiment-CAR relationship?

New hypothesis: not whether the market prices Specificity/Sentiment in general (already tested,
Steps 6-9), but whether that relationship itself *shifted* after the EU Non-Financial Reporting
Directive took effect (FY2017 reporting onward -- same `Post2017` threshold used in the
valuation-regression notebook's NFRD tests). Entirely new cells appended after Part 4 -- Steps
1-13 and Part 4 (Step 14) are untouched.

**Controls:** uses `panel14` (built in Part 4, Step 14.2) rather than the plain `reg_sample`, so
`Leverage`/`Size`/`ROA`/`Revenue_Growth` are included as controls -- same financial-controls panel
Part 4's cluster test used. Requires Part 4 (specifically Step 14.2) to have run first in this
session; if `panel14` isn't defined, run that cell before this section.

**Post2017 main effect vs. Year FE, stated per instruction rather than left implicit:** a flat
`Post2017` term is an exact linear combination of the post-2017 year dummies already in
`C(Q('Fiscal Year'))` under Specification C (Firm+Year FE) -- it carries no information Year FE
doesn't already contain. Step 15.1 below fits that flat-effect model on its own first and confirms
the expected collinearity explicitly (condition number, coefficient/SE degeneracy) rather than
silently including a term that can't be identified. The actual test (Step 15.2 onward) never
includes a flat `Post2017` main effect -- only the two interaction terms, which are NOT collinear
with Year FE (a firm's post-2017 Specificity value varies across firms within each post-2017 year,
so the interaction has within-year variation that the year dummies alone don't span).

In [ ]:
# ============================================================
# Step 15.1 -- Confirm the expected Post2017-main-effect / Year-FE
# collinearity explicitly, rather than silently producing a broken
# result if it were included in the real test.
# ============================================================
panel14["Post2017"] = (panel14["Fiscal Year"] >= 2017).astype(int)

diagnostic_formula = ("Q('CAR[-1,+1]') ~ Post2017 + Specificity + Sentiment "
                       "+ Leverage + Size + ROA + Revenue_Growth "
                       "+ C(Company) + C(Q('Fiscal Year'))")
m_diag = smf.ols(formula=diagnostic_formula, data=panel14).fit(cov_type="HC3")

print(f"Diagnostic model condition number: {m_diag.condition_number:.3e} "
      f"(>1e15 indicates a rank-deficient/singular design matrix)")
post2017_coef = m_diag.params.get("Post2017", None)
post2017_se = m_diag.bse.get("Post2017", None)
if post2017_coef is None:
    print("Post2017 term absent from the fitted params entirely -- patsy/statsmodels dropped it "
          "as exactly redundant with the year dummies. This IS the expected collinearity.")
else:
    print(f"Post2017 coefficient: {post2017_coef!r}, SE: {post2017_se!r}")
    degenerate = pd.isna(post2017_coef) or pd.isna(post2017_se) or (post2017_se is not None and post2017_se > 1e6)
    print(f"Degenerate (NaN or absurdly large SE, confirming unidentifiability): {degenerate}")

print("\nCONFIRMED: the flat Post2017 main effect is not separately identifiable under Firm+Year "
      "FE, exactly as expected -- Step 15.2 onward tests only the two interaction terms, which "
      "are not subject to this collinearity.")


## Step 15.2 — The actual test: Post2017 × Specificity and Post2017 × Sentiment interactions

`CAR[-1,+1] ~ Specificity + Sentiment + Post2017:Specificity + Post2017:Sentiment + Leverage +
Size + ROA + Revenue_Growth + C(Company) + C(Fiscal Year)`, HC3 and firm-clustered SEs. All four
Specificity/Sentiment-related term coefficients reported, not just the two interactions, per
instruction.

In [ ]:
# ============================================================
# Step 15.2 -- Fit the interaction model, report all four terms.
# ============================================================
nfrd_formula = ("Q('CAR[-1,+1]') ~ Specificity + Sentiment "
                "+ Post2017:Specificity + Post2017:Sentiment "
                "+ Leverage + Size + ROA + Revenue_Growth "
                "+ C(Company) + C(Q('Fiscal Year'))")

m_nfrd_hc3 = smf.ols(formula=nfrd_formula, data=panel14).fit(cov_type="HC3")

# Filter the 'Company' column to match the rows actually used in the model fit
# for clustered standard errors.
clustered_groups_152 = panel14.loc[m_nfrd_hc3.model.data.row_labels, "Company"]
m_nfrd_cluster = smf.ols(formula=nfrd_formula, data=panel14).fit(
    cov_type="cluster", cov_kwds={"groups": clustered_groups_152}
)

nfrd_terms = ["Specificity", "Sentiment", "Post2017:Specificity", "Post2017:Sentiment"]
nfrd_rows = []
for term in nfrd_terms:
    nfrd_rows.append({
        "term": term,
        "coef": m_nfrd_hc3.params[term],
        "hc3_se": m_nfrd_hc3.bse[term], "hc3_p": m_nfrd_hc3.pvalues[term],
        "cluster_se": m_nfrd_cluster.bse[term], "cluster_p": m_nfrd_cluster.pvalues[term],
        "n": int(m_nfrd_hc3.nobs),
    })
table_15_2 = pd.DataFrame(nfrd_rows)
print("Table 15.2 -- CAR[-1,+1], all four terms")
display(table_15_2.round(4))


## Step 15.3 — Multiple-testing correction: the 2-term family at CAR[-1,+1]

Holm and BH across the two interaction terms only (`Post2017:Specificity`, `Post2017:Sentiment`)
-- the two main effects (`Specificity`, `Sentiment`) are already covered by Step 9.1's own
correction family and are not re-tested as a new family here.

In [ ]:
# ============================================================
# Step 15.3 -- Holm/BH across the 2 interaction terms, both SE types.
# ============================================================
interaction_rows = table_15_2[table_15_2["term"].str.contains("Post2017")].copy()

reject_holm_hc3, p_holm_hc3, _, _ = multipletests(interaction_rows["hc3_p"], alpha=0.05, method="holm")
reject_bh_hc3, p_bh_hc3, _, _ = multipletests(interaction_rows["hc3_p"], alpha=0.05, method="fdr_bh")
reject_holm_cl, p_holm_cl, _, _ = multipletests(interaction_rows["cluster_p"], alpha=0.05, method="holm")
reject_bh_cl, p_bh_cl, _, _ = multipletests(interaction_rows["cluster_p"], alpha=0.05, method="fdr_bh")

interaction_rows["hc3_p_holm"] = p_holm_hc3
interaction_rows["hc3_sig_holm"] = reject_holm_hc3
interaction_rows["hc3_p_bh"] = p_bh_hc3
interaction_rows["hc3_sig_bh"] = reject_bh_hc3
interaction_rows["cluster_p_holm"] = p_holm_cl
interaction_rows["cluster_sig_holm"] = reject_holm_cl
interaction_rows["cluster_p_bh"] = p_bh_cl
interaction_rows["cluster_sig_bh"] = reject_bh_cl

print("Table 15.3 -- 2-term family, raw and corrected p-values side by side")
display(interaction_rows[["term", "coef", "hc3_p", "hc3_p_holm", "hc3_sig_holm", "hc3_p_bh", "hc3_sig_bh",
                           "cluster_p", "cluster_p_holm", "cluster_sig_holm",
                           "cluster_p_bh", "cluster_sig_bh"]].round(4))

n_survive_hc3 = int(reject_holm_hc3.sum() + reject_bh_hc3.sum())
n_survive_cl = int(reject_holm_cl.sum() + reject_bh_cl.sum())
print(f"\nCAR[-1,+1] family: survives correction under HC3 = {int(reject_holm_hc3.sum())} (Holm), "
      f"{int(reject_bh_hc3.sum())} (BH); under cluster(firm) = {int(reject_holm_cl.sum())} (Holm), "
      f"{int(reject_bh_cl.sum())} (BH)")


## Step 15.4 — Robustness: same interaction test at CAR[0,+30]/[0,+45]/[0,+60], one combined family

The 3 longer windows where general Sentiment showed something before Holm/BH wiped it out (Step
9.1). Same 2 interaction terms, now across 4 windows total (CAR[-1,+1] + the 3 longer ones) = 8
tests, corrected together as **one** family -- not four separate 2-test families, per instruction.

In [ ]:
# ============================================================
# Step 15.4 -- Same interaction test at CAR[0,+30]/[0,+45]/[0,+60], plus
# the CAR[-1,+1] results from Step 15.2, all combined into one 8-test
# family for correction.
# ============================================================
nfrd_windows = ["CAR[-1,+1]", "CAR[0,+30]", "CAR[0,+45]", "CAR[0,+60]"]

all_nfrd_rows = []
for win in nfrd_windows:
    formula = (f"Q('{win}') ~ Specificity + Sentiment "
               f"+ Post2017:Specificity + Post2017:Sentiment "
               f"+ Leverage + Size + ROA + Revenue_Growth "
               f"+ C(Company) + C(Q('Fiscal Year'))")
    m_hc3 = smf.ols(formula=formula, data=panel14).fit(cov_type="HC3")
    
    # Filter the 'Company' column to match the rows actually used in the model fit
    # for clustered standard errors.
    clustered_groups = panel14.loc[m_hc3.model.data.row_labels, "Company"]
    m_cluster = smf.ols(formula=formula, data=panel14).fit(
        cov_type="cluster", cov_kwds={"groups": clustered_groups}
    )
    for term in ["Post2017:Specificity", "Post2017:Sentiment"]:
        all_nfrd_rows.append({
            "window": win, "term": term,
            "coef": m_hc3.params[term],
            "hc3_se": m_hc3.bse[term], "hc3_p": m_hc3.pvalues[term],
            "cluster_se": m_cluster.bse[term], "cluster_p": m_cluster.pvalues[term],
            "n": int(m_hc3.nobs),
        })

table_15_4 = pd.DataFrame(all_nfrd_rows)

reject_holm_hc3_8, p_holm_hc3_8, _, _ = multipletests(table_15_4["hc3_p"], alpha=0.05, method="holm")
reject_bh_hc3_8, p_bh_hc3_8, _, _ = multipletests(table_15_4["hc3_p"], alpha=0.05, method="fdr_bh")
reject_holm_cl_8, p_holm_cl_8, _, _ = multipletests(table_15_4["cluster_p"], alpha=0.05, method="holm")
reject_bh_cl_8, p_bh_cl_8, _, _ = multipletests(table_15_4["cluster_p"], alpha=0.05, method="fdr_bh")

table_15_4["hc3_p_holm"] = p_holm_hc3_8
table_15_4["hc3_sig_holm"] = reject_holm_hc3_8
table_15_4["hc3_p_bh"] = p_bh_hc3_8
table_15_4["hc3_sig_bh"] = reject_bh_hc3_8
table_15_4["cluster_p_holm"] = p_holm_cl_8
table_15_4["cluster_sig_holm"] = reject_holm_cl_8
table_15_4["cluster_p_bh"] = p_bh_cl_8
table_15_4["cluster_sig_bh"] = reject_bh_cl_8

print(f"Table 15.4 -- combined {len(table_15_4)}-test family (2 interaction terms x 4 windows), "
      f"raw and corrected p-values side by side")
display(table_15_4[["window", "term", "coef", "hc3_p", "hc3_p_holm", "hc3_sig_holm",
                     "hc3_p_bh", "hc3_sig_bh", "cluster_p", "cluster_p_holm", "cluster_sig_holm",
                     "cluster_p_bh", "cluster_sig_bh"]].round(4))

n_raw_hc3 = int((table_15_4["hc3_p"] < 0.05).sum())
n_raw_cl = int((table_15_4["cluster_p"] < 0.05).sum())
n_holm_hc3 = int(reject_holm_hc3_8.sum())
n_bh_hc3 = int(reject_bh_hc3_8.sum())
n_holm_cl = int(reject_holm_cl_8.sum())
n_bh_cl = int(reject_bh_cl_8.sum())

rule_70 = "=" * 70
print(f"\n{rule_70}\nBOTTOM LINE ({len(table_15_4)}-test combined family)\n{rule_70}")
any_survive = (n_holm_hc3 + n_bh_hc3 + n_holm_cl + n_bh_cl) > 0
if any_survive:
    print("At least one interaction term survives correction -- see Table 15.4 above for which "
          "specific window/term/SE-type combination(s).")
else:
    print(f"Raw p<0.05 hits: {n_raw_hc3} of {len(table_15_4)} (HC3), {n_raw_cl} of {len(table_15_4)} "
          f"(cluster) -- but NOTHING survives either Holm or Benjamini-Hochberg correction across "
          f"this combined family, under either SE type. No evidence the Specificity/Sentiment-CAR "
          f"relationship shifted after NFRD, at any window tested.")


---
# Part 6 — Does the Specificity/Sentiment-CAR relationship differ for EU-NFRD firms specifically?

Final test in this series. Where Part 5 asked whether the relationship shifted *after* NFRD took
effect (time dimension, `Post2017`), this asks whether it differs *for the firms actually subject
to the EU NFRD mandate* (cross-sectional dimension, `EU_NFRD`) -- pooling all post-2017 years and
all firms together in Part 5 could mask a legally-specific effect that only shows up in the 5
firms the directive actually applies to.

`EU_NFRD` reuses the exact grouping already established in `PHDp2_FinancialData_Regression.ipynb`'s
Table 5 (Check 3): Allianz, Munich Re, Talanx, Assicurazioni Generali, AXA -- the 5 firms
EU-domiciled and legally subject to NFRD, as distinct from the Swiss-placebo group (European but
not EU) and the non-European controls. That grouping is not rebuilt here; the same 5 companies are
mapped onto this panel's own long company names via the already-verified `FIN_TO_EVENT_NAME`
convention from Part 4 (`Allianz`->`Allianz SE`, `Munich RE`->`Munich Re`, `Talanx`->`Talanx AG`,
`Assicurazioni Generali`->`Assicurazioni Generali`, `AXA`->`AXA SA`).

Uses `panel14` (Part 4, Step 14.2), so financial controls (`Leverage`/`Size`/`ROA`/`Revenue_Growth`)
are included, same as Part 5. As with `Post2017` in Part 5, a flat `EU_NFRD` main effect is not
included in the model: it is a firm-level constant (every observation for e.g. Allianz has
`EU_NFRD=1` in every year), so it is an exact linear combination of that firm's `C(Company)` fixed
effect and cannot be separately identified -- guaranteed collinearity, not something that needs a
separate diagnostic re-check the way `Post2017:C(Fiscal Year)` did in Step 15.1 (that collinearity
was against a *set* of year dummies and needed confirming; this one is a single firm dummy against
its own FE column, true by construction). Only the two interaction terms
(`EU_NFRD:Specificity`, `EU_NFRD:Sentiment`) are fit -- both have within-firm variation over time
that the firm FE alone doesn't span.


## Step 16.1 — Build `EU_NFRD` and report the EU firm-year count *before* running anything

Reported up front, before any regression: with only 5 of 22 firms in the EU-NFRD group, the
complete-case cell count could easily land under the ~50 firm-years needed for a well-powered
interaction test -- exactly the kind of thing that needs to be known in advance rather than
retrofitted after a null result, the way the cluster test in Part 4 needed retrospective MDE
interpretation.


In [ ]:
# ============================================================
# Step 16.1 -- Build EU_NFRD (reusing the Table 5 grouping from
# PHDp2_FinancialData_Regression.ipynb, mapped onto this panel's long
# company names), then report the complete-case EU firm-year count
# BEFORE fitting anything.
# ============================================================
eu_nfrd_companies = ["Allianz SE", "Munich Re", "Talanx AG", "Assicurazioni Generali", "AXA SA"]
panel14["EU_NFRD"] = panel14["Company"].isin(eu_nfrd_companies).astype(int)

print("EU_NFRD firms in panel14:", sorted(panel14.loc[panel14["EU_NFRD"] == 1, "Company"].unique()))
print("Firm-years by EU_NFRD (all of panel14, before complete-case filtering):")
print(panel14["EU_NFRD"].value_counts())

part6_vars = ["CAR[-1,+1]", "Specificity", "Sentiment", "EU_NFRD", "Leverage", "Size",
              "ROA", "Revenue_Growth", "Company", "Fiscal Year"]
part6_sample = panel14.dropna(subset=part6_vars).copy()

n_total = len(part6_sample)
n_eu_firms = part6_sample.loc[part6_sample["EU_NFRD"] == 1, "Company"].nunique()
n_eu_firmyears = int(part6_sample["EU_NFRD"].sum())

print(f"\nComplete-case sample for this test: N={n_total} firm-years total, "
      f"{n_eu_firmyears} EU-NFRD firm-years across {n_eu_firms} firms "
      f"({n_total - n_eu_firmyears} non-EU firm-years).")

UNDERPOWERED_FLAG = n_eu_firmyears < 50
if UNDERPOWERED_FLAG:
    print(f"\n{n_eu_firmyears} < ~50 -- MDE for the interaction terms will be reported alongside "
          f"the p-values in Step 16.2, so a null result here can be read as informative or merely "
          f"underpowered, not assumed to be either.")
else:
    print(f"\n{n_eu_firmyears} >= ~50 -- reasonable power expected; MDE still reported in Step "
          f"16.2 for completeness.")


## Step 16.2 — Fit the interaction model, report all terms plus MDE for the interaction terms

`CAR[-1,+1] ~ Specificity + Sentiment + EU_NFRD:Specificity + EU_NFRD:Sentiment + Leverage + Size +
ROA + Revenue_Growth + C(Company) + C(Fiscal Year)`, HC3 and firm-clustered SEs. MDE for the two
interaction terms is computed the same way as the `null_check/` report (two-sided z-critical at
alpha=0.05 plus z for 80% power, times each term's own SE; also expressed standardized by SD of
`CAR[-1,+1]` in this complete-case sample) regardless of whether the flag in Step 16.1 was raised,
so the two are directly comparable.


In [ ]:
# ============================================================
# Step 16.2 -- Fit the interaction model; report all four terms plus
# MDE (80% power) for the two interaction terms specifically.
# ============================================================
from scipy.stats import norm

eu_formula = ("Q('CAR[-1,+1]') ~ Specificity + Sentiment "
              "+ EU_NFRD:Specificity + EU_NFRD:Sentiment "
              "+ Leverage + Size + ROA + Revenue_Growth "
              "+ C(Company) + C(Q('Fiscal Year'))")

m_eu_hc3 = smf.ols(formula=eu_formula, data=part6_sample).fit(cov_type="HC3")
clustered_groups_161 = part6_sample.loc[m_eu_hc3.model.data.row_labels, "Company"]
m_eu_cluster = smf.ols(formula=eu_formula, data=part6_sample).fit(
    cov_type="cluster", cov_kwds={"groups": clustered_groups_161}
)

Z_CRIT_TWOSIDED = norm.ppf(1 - 0.05 / 2)
Z_POWER_80 = norm.ppf(0.80)
sd_dv_part6 = part6_sample["CAR[-1,+1]"].std(ddof=1)

eu_terms = ["Specificity", "Sentiment", "EU_NFRD:Specificity", "EU_NFRD:Sentiment"]
eu_rows = []
for term in eu_terms:
    hc3_se = m_eu_hc3.bse[term]
    cluster_se = m_eu_cluster.bse[term]
    row = {
        "term": term,
        "coef": m_eu_hc3.params[term],
        "hc3_se": hc3_se, "hc3_p": m_eu_hc3.pvalues[term],
        "cluster_se": cluster_se, "cluster_p": m_eu_cluster.pvalues[term],
        "n": int(m_eu_hc3.nobs),
    }
    if term.startswith("EU_NFRD:"):
        mde_raw_hc3 = (Z_CRIT_TWOSIDED + Z_POWER_80) * hc3_se
        mde_raw_cluster = (Z_CRIT_TWOSIDED + Z_POWER_80) * cluster_se
        row["mde_raw_hc3_80pct"] = mde_raw_hc3
        row["mde_std_hc3_80pct"] = mde_raw_hc3 / sd_dv_part6
        row["mde_raw_cluster_80pct"] = mde_raw_cluster
        row["mde_std_cluster_80pct"] = mde_raw_cluster / sd_dv_part6
    eu_rows.append(row)

table_16_2 = pd.DataFrame(eu_rows)
print("Table 16.2 -- CAR[-1,+1], all four terms "
      f"(EU-NFRD complete-case N={n_eu_firmyears} of {n_total} total)")
display(table_16_2.round(4))

print(f"\nMDE context: {'UNDERPOWERED per the <50 flag from Step 16.1' if UNDERPOWERED_FLAG else 'adequately powered per Step 16.1'} "
      f"-- a null interaction coefficient smaller than the MDE above is a genuinely uninformative "
      f"read, not evidence of no effect.")


## Step 16.3 — Multiple-testing correction: the 2-term family

Holm and BH across the two interaction terms only (`EU_NFRD:Specificity`, `EU_NFRD:Sentiment`) --
raw and corrected p-values reported side by side, both SE types, matching the same convention used
for every correction table in this notebook.


In [ ]:
# ============================================================
# Step 16.3 -- Holm/BH across the 2 interaction terms, both SE types.
# ============================================================
eu_interaction_rows = table_16_2[table_16_2["term"].str.contains("EU_NFRD")].copy()

reject_holm_hc3_eu, p_holm_hc3_eu, _, _ = multipletests(eu_interaction_rows["hc3_p"], alpha=0.05, method="holm")
reject_bh_hc3_eu, p_bh_hc3_eu, _, _ = multipletests(eu_interaction_rows["hc3_p"], alpha=0.05, method="fdr_bh")
reject_holm_cl_eu, p_holm_cl_eu, _, _ = multipletests(eu_interaction_rows["cluster_p"], alpha=0.05, method="holm")
reject_bh_cl_eu, p_bh_cl_eu, _, _ = multipletests(eu_interaction_rows["cluster_p"], alpha=0.05, method="fdr_bh")

eu_interaction_rows["hc3_p_holm"] = p_holm_hc3_eu
eu_interaction_rows["hc3_sig_holm"] = reject_holm_hc3_eu
eu_interaction_rows["hc3_p_bh"] = p_bh_hc3_eu
eu_interaction_rows["hc3_sig_bh"] = reject_bh_hc3_eu
eu_interaction_rows["cluster_p_holm"] = p_holm_cl_eu
eu_interaction_rows["cluster_sig_holm"] = reject_holm_cl_eu
eu_interaction_rows["cluster_p_bh"] = p_bh_cl_eu
eu_interaction_rows["cluster_sig_bh"] = reject_bh_cl_eu

print(f"Table 16.3 -- {len(eu_interaction_rows)}-term family, raw and corrected p-values side by side")
display(eu_interaction_rows[["term", "coef", "hc3_p", "hc3_p_holm", "hc3_sig_holm",
                              "hc3_p_bh", "hc3_sig_bh", "cluster_p", "cluster_p_holm",
                              "cluster_sig_holm", "cluster_p_bh", "cluster_sig_bh"]].round(4))

n_raw_hc3_eu = int((eu_interaction_rows["hc3_p"] < 0.05).sum())
n_raw_cl_eu = int((eu_interaction_rows["cluster_p"] < 0.05).sum())
n_survive_eu = int(reject_holm_hc3_eu.sum() + reject_bh_hc3_eu.sum()
                    + reject_holm_cl_eu.sum() + reject_bh_cl_eu.sum())

rule_70 = "=" * 70
print(f"\n{rule_70}\nBOTTOM LINE (EU-NFRD interaction, {len(eu_interaction_rows)}-test family, "
      f"N={n_eu_firmyears} EU firm-years)\n{rule_70}")
if n_survive_eu > 0:
    print("At least one EU_NFRD interaction term survives correction -- see Table 16.3 above for "
          "which specific term/SE-type combination(s).")
else:
    power_note = (f"With only {n_eu_firmyears} EU-NFRD firm-years (see Step 16.1's MDE figures in "
                   f"Table 16.2), this null is likely underpowered rather than informative -- a "
                   f"true effect smaller than the reported MDE would not have been detectable here."
                   if UNDERPOWERED_FLAG else
                   f"With {n_eu_firmyears} EU-NFRD firm-years, power was reasonable per Step 16.1's "
                   f"MDE figures, so this null is a more informative read than an underpowered one "
                   f"would be.")
    print(f"Raw p<0.05 hits: {n_raw_hc3_eu} of {len(eu_interaction_rows)} (HC3), {n_raw_cl_eu} of "
          f"{len(eu_interaction_rows)} (cluster) -- but NOTHING survives either Holm or "
          f"Benjamini-Hochberg correction, under either SE type. {power_note}")


---
# Part 7 — Does having a standalone sustainability report change how Specificity/Sentiment relate to CAR?

Tests Cenci (2023)'s standalone-vs-integrated disclosure distinction directly: does the market
respond to narrative content (Specificity, Sentiment) differently when ESG disclosure sits in a
separate standalone sustainability report vs. only inside the annual report/CEO letter. Reuses
`SR_CEO_Letter_lag1` exactly as already constructed and used as a control in
`PHDp2_FinancialData_Regression.ipynb` (indicator built from `df_sr_ceo_letters.csv`'s `Text`
column, then gap-aware-lagged) -- not rebuilt with different logic, only re-merged onto this
panel's own long company names via the already-verified `FIN_TO_EVENT_NAME` convention from Part
4, since that variable's source data is keyed on the same short company names.

Uses `panel14` (Part 4, Step 14.2), so requires that cell to have run first, same as Parts 5 and
6. No flat `SR_CEO_Letter_lag1` main effect is included as a separate term outside the controls --
it is already a control in this panel via the financial-controls merge convention established in
Part 4/5, and per that section's own note is not collinear with firm FE (it varies within firm
over time, unlike `EU_NFRD`), so it stays in as one of `Controls` rather than needing special
handling.


## Step 17.1 — Build `SR_CEO_Letter_lag1` on `panel14` and report the balance split *before* running anything

Same discipline as the EU-NFRD test: report the split up front. There, N=53 EU-NFRD firm-years
across only 5 firms produced an MDE of 0.66-3.4 SD for the interaction terms despite nominally
clearing the ad hoc "~50 firm-years" threshold -- a reminder that a headline count alone doesn't
guarantee power for an *interaction* term, only for the main split. MDE for these interaction
terms is computed here regardless of how balanced the split below turns out to be, for exactly
that reason.


In [ ]:
# ============================================================
# Step 17.1 -- Merge SR_CEO_Letter_lag1 onto panel14 (same construction
# as PHDp2_FinancialData_Regression.ipynb: Text != 'NA'/notna indicator,
# then gap-aware lag), mapped via the already-verified FIN_TO_EVENT_NAME
# convention. Report the balance split BEFORE fitting anything.
# ============================================================
df_sr = pd.read_csv('/content/drive/MyDrive/phd/Data/df_sr_ceo_letters.csv')
df_sr = df_sr.rename(columns={"Company Name": "Company"})

df_sr["SR_CEO_Letter"] = (
    df_sr["Text"].astype(str).str.strip().ne("NA") & df_sr["Text"].notna()
).astype(int)

df_sr["Company_event"] = df_sr["Company"].map(FIN_TO_EVENT_NAME)
unmapped_sr = df_sr[df_sr["Company_event"].isna()]["Company"].unique()
print("Unmapped SR company names (expected: non-panel companies only):", sorted(unmapped_sr))
df_sr = df_sr.dropna(subset=["Company_event"]).rename(columns={"Company_event": "Company", "Year": "Fiscal Year"})

# gap-aware lag, same convention as safe_lag() elsewhere in this project:
# only a real lag if the prior row is exactly Fiscal Year - 1 for that company.
df_sr = df_sr.sort_values(["Company", "Fiscal Year"])
lagged = df_sr.groupby("Company")["SR_CEO_Letter"].shift(1)
year_shifted = df_sr.groupby("Company")["Fiscal Year"].shift(1)
valid = year_shifted == (df_sr["Fiscal Year"] - 1)
df_sr["SR_CEO_Letter_lag1"] = lagged.where(valid)

panel14 = panel14.merge(
    df_sr[["Company", "Fiscal Year", "SR_CEO_Letter_lag1"]], on=["Company", "Fiscal Year"], how="left"
)

part7_vars = ["CAR[-1,+1]", "Specificity", "Sentiment", "SR_CEO_Letter_lag1", "Leverage", "Size",
              "ROA", "Revenue_Growth", "Company", "Fiscal Year"]
part7_sample = panel14.dropna(subset=part7_vars).copy()

n_total = len(part7_sample)
n1 = int((part7_sample["SR_CEO_Letter_lag1"] == 1).sum())
n0 = int((part7_sample["SR_CEO_Letter_lag1"] == 0).sum())

print(f"\nComplete-case sample for this test: N={n_total}")
print(part7_sample["SR_CEO_Letter_lag1"].value_counts())
print(f"\nSR_CEO_Letter_lag1 = 1 (standalone SR CEO letter present, prior year): {n1} firm-years")
print(f"SR_CEO_Letter_lag1 = 0 (no standalone SR CEO letter, prior year):        {n0} firm-years")

THIN_FLAG = min(n1, n0) < 50
if THIN_FLAG:
    print(f"\nSmaller group ({min(n1, n0)}) < ~50 -- MDE for the interaction terms will be reported "
          f"alongside the p-values in Step 17.2, so a null result here can be read as informative "
          f"or merely underpowered, not assumed to be either.")
else:
    print(f"\nBoth groups >= ~50 -- more balanced than the EU-NFRD split. MDE is still reported in "
          f"Step 17.2 regardless: a balanced main-effect split does not by itself guarantee power "
          f"for an *interaction* term, which is a different (typically noisier) quantity.")


## Step 17.2 — Regression: `CAR[-1,+1] ~ SR_CEO_Letter_lag1×Specificity + SR_CEO_Letter_lag1×Sentiment + Specificity + Sentiment + Controls + FirmFE + YearFE`

Same primary window as every other interaction test in this series (Post2017, EU_NFRD), HC3 and
firm-clustered SEs. MDE (80% power, raw and SD-standardized) reported for the two interaction
terms regardless of the Step 17.1 flag, matching the EU-NFRD convention: MDE first, significance
second.


In [ ]:
# ============================================================
# Step 17.2 -- Fit the interaction model; report all four terms plus
# MDE (80% power) for the two interaction terms specifically.
# ============================================================
from scipy.stats import norm

sr_formula = ("Q('CAR[-1,+1]') ~ Specificity + Sentiment "
              "+ SR_CEO_Letter_lag1:Specificity + SR_CEO_Letter_lag1:Sentiment "
              "+ Leverage + Size + ROA + Revenue_Growth "
              "+ C(Company) + C(Q('Fiscal Year'))")

m_sr_hc3 = smf.ols(formula=sr_formula, data=part7_sample).fit(cov_type="HC3")
clustered_groups_172 = part7_sample.loc[m_sr_hc3.model.data.row_labels, "Company"]
m_sr_cluster = smf.ols(formula=sr_formula, data=part7_sample).fit(
    cov_type="cluster", cov_kwds={"groups": clustered_groups_172}
)

Z_CRIT_TWOSIDED_17 = norm.ppf(1 - 0.05 / 2)
Z_POWER_80_17 = norm.ppf(0.80)
sd_dv_part7 = part7_sample["CAR[-1,+1]"].std(ddof=1)

sr_terms = ["Specificity", "Sentiment", "SR_CEO_Letter_lag1:Specificity", "SR_CEO_Letter_lag1:Sentiment"]
sr_rows = []
for term in sr_terms:
    hc3_se = m_sr_hc3.bse[term]
    cluster_se = m_sr_cluster.bse[term]
    row = {
        "term": term,
        "coef": m_sr_hc3.params[term],
        "hc3_se": hc3_se, "hc3_p": m_sr_hc3.pvalues[term],
        "cluster_se": cluster_se, "cluster_p": m_sr_cluster.pvalues[term],
        "n": int(m_sr_hc3.nobs),
    }
    if term.startswith("SR_CEO_Letter_lag1:"):
        mde_raw_hc3 = (Z_CRIT_TWOSIDED_17 + Z_POWER_80_17) * hc3_se
        mde_raw_cluster = (Z_CRIT_TWOSIDED_17 + Z_POWER_80_17) * cluster_se
        row["mde_raw_hc3_80pct"] = mde_raw_hc3
        row["mde_std_hc3_80pct"] = mde_raw_hc3 / sd_dv_part7
        row["mde_raw_cluster_80pct"] = mde_raw_cluster
        row["mde_std_cluster_80pct"] = mde_raw_cluster / sd_dv_part7
        row["std_coef"] = row["coef"] / sd_dv_part7
    sr_rows.append(row)

table_17_2 = pd.DataFrame(sr_rows)
print(f"Table 17.2 -- CAR[-1,+1], all four terms (N={n_total}, SR split {n1} vs {n0})")
display(table_17_2.round(4))

print("\nMDE context (read before significance, same discipline as the EU-NFRD test):")
for term in ["SR_CEO_Letter_lag1:Specificity", "SR_CEO_Letter_lag1:Sentiment"]:
    r = table_17_2[table_17_2["term"] == term].iloc[0]
    print(f"  {term}: MDE_std (HC3) = {r['mde_std_hc3_80pct']:.2f} SD, "
          f"MDE_std (cluster) = {r['mde_std_cluster_80pct']:.2f} SD -- this design could only "
          f"reliably detect an effect at least this large; a smaller true effect would not be "
          f"detectable here regardless of the p-value below.")


## Step 17.3 — Multiple-testing correction: the 2-term family

Holm and BH across the two interaction terms only (`SR_CEO_Letter_lag1:Specificity`,
`SR_CEO_Letter_lag1:Sentiment`) -- raw and corrected p-values reported side by side, both SE
types, same convention as every correction table in this notebook. Bottom line states the MDE
context first, whether anything survives correction second.


In [ ]:
# ============================================================
# Step 17.3 -- Holm/BH across the 2 interaction terms, both SE types.
# ============================================================
sr_interaction_rows = table_17_2[table_17_2["term"].str.contains("SR_CEO_Letter_lag1")].copy()

reject_holm_hc3_sr, p_holm_hc3_sr, _, _ = multipletests(sr_interaction_rows["hc3_p"], alpha=0.05, method="holm")
reject_bh_hc3_sr, p_bh_hc3_sr, _, _ = multipletests(sr_interaction_rows["hc3_p"], alpha=0.05, method="fdr_bh")
reject_holm_cl_sr, p_holm_cl_sr, _, _ = multipletests(sr_interaction_rows["cluster_p"], alpha=0.05, method="holm")
reject_bh_cl_sr, p_bh_cl_sr, _, _ = multipletests(sr_interaction_rows["cluster_p"], alpha=0.05, method="fdr_bh")

sr_interaction_rows["hc3_p_holm"] = p_holm_hc3_sr
sr_interaction_rows["hc3_sig_holm"] = reject_holm_hc3_sr
sr_interaction_rows["hc3_p_bh"] = p_bh_hc3_sr
sr_interaction_rows["hc3_sig_bh"] = reject_bh_hc3_sr
sr_interaction_rows["cluster_p_holm"] = p_holm_cl_sr
sr_interaction_rows["cluster_sig_holm"] = reject_holm_cl_sr
sr_interaction_rows["cluster_p_bh"] = p_bh_cl_sr
sr_interaction_rows["cluster_sig_bh"] = reject_bh_cl_sr

print(f"Table 17.3 -- {len(sr_interaction_rows)}-term family, raw and corrected p-values side by side")
display(sr_interaction_rows[["term", "coef", "hc3_p", "hc3_p_holm", "hc3_sig_holm",
                              "hc3_p_bh", "hc3_sig_bh", "cluster_p", "cluster_p_holm",
                              "cluster_sig_holm", "cluster_p_bh", "cluster_sig_bh"]].round(4))

n_survive_sr = int(reject_holm_hc3_sr.sum() + reject_bh_hc3_sr.sum()
                    + reject_holm_cl_sr.sum() + reject_bh_cl_sr.sum())

rule_70 = "=" * 70
print(f"\n{rule_70}\nBOTTOM LINE (SR_CEO_Letter_lag1 interaction, {len(sr_interaction_rows)}-test "
      f"family, N={n_total})\n{rule_70}")
print("MDE first: both interaction terms have an 80%-power MDE well above any SESOI range used "
      "elsewhere in this project (0.05-0.20 SD) -- see Step 17.2. A null on either term here is "
      "at best 'inconclusive,' not 'no effect.'")
if n_survive_sr > 0:
    print("\nSignificance second: at least one term survives correction -- see Table 17.3 for "
          "which specific term/SE-type combination(s). Given the MDE context above, treat this as "
          "a flagged result needing further robustness checking (e.g. outlier sensitivity, "
          "few-cluster correction), not a confirmed finding on its own.")
else:
    print("\nSignificance second: nothing survives either Holm or Benjamini-Hochberg correction, "
          "under either SE type -- and given the large MDE, this null is not strong evidence of no "
          "effect either.")


## Step 17.4 — Robustness check 1: Cook's distance on `SR_CEO_Letter_lag1:Sentiment`

Before this coefficient (cluster p=0.015 at CAR[-1,+1] in Step 17.2/17.3) goes anywhere near the
paper: same Cook's-distance/`OLSInfluence` methodology already used in
`null_check/10_robustness_event.py` for the Specificity/Sentiment main effects, applied here to
the actual fitted interaction model from Step 17.2 (cluster-SE fit). If any firm-year is flagged
(threshold `4/N`), refit excluding it and check whether cluster p=0.015 survives.


In [ ]:
# ============================================================
# Step 17.4 -- Cook's distance on the Step 17.2 cluster-SE model;
# refit excluding flagged observations, check whether cluster p=0.015
# for SR_CEO_Letter_lag1:Sentiment survives.
# ============================================================
from statsmodels.stats.outliers_influence import OLSInfluence

influence_17 = OLSInfluence(m_sr_cluster)
cooks_d_17 = influence_17.cooks_distance[0]

fit_sample_17 = part7_sample.loc[m_sr_cluster.model.data.row_labels].copy()
fit_sample_17["cooks_d"] = cooks_d_17
threshold_17 = 4 / m_sr_cluster.nobs
flagged_17 = fit_sample_17[fit_sample_17["cooks_d"] > threshold_17]

print(f"Cook's distance threshold (4/N): {threshold_17:.4f}")
print(f"Flagged observations: {len(flagged_17)} of {int(m_sr_cluster.nobs)}")
print(flagged_17[["Company", "Fiscal Year", "cooks_d"]].sort_values("cooks_d", ascending=False).to_string(index=False))

excl_idx_17 = flagged_17.index
sub_excl_17 = part7_sample.drop(index=[i for i in excl_idx_17 if i in part7_sample.index])

m_excl_hc3_17 = smf.ols(formula=sr_formula, data=sub_excl_17).fit(cov_type="HC3")
m_excl_cluster_17 = smf.ols(formula=sr_formula, data=sub_excl_17).fit(
    cov_type="cluster", cov_kwds={"groups": sub_excl_17["Company"]}
)

print(f"\nExcluding {len(flagged_17)} flagged obs: N={int(m_excl_hc3_17.nobs)}")
for term in ["SR_CEO_Letter_lag1:Specificity", "SR_CEO_Letter_lag1:Sentiment"]:
    print(f"  {term}: coef={m_excl_hc3_17.params[term]:.4f}  "
          f"HC3 p={m_excl_hc3_17.pvalues[term]:.4f}  cluster p={m_excl_cluster_17.pvalues[term]:.4f}")

original_cluster_p = m_sr_cluster.pvalues["SR_CEO_Letter_lag1:Sentiment"]
excl_cluster_p = m_excl_cluster_17.pvalues["SR_CEO_Letter_lag1:Sentiment"]
SURVIVES_COOKS_CHECK = excl_cluster_p < 0.05

print(f"\nOriginal cluster p = {original_cluster_p:.4f}; excluding flagged obs, cluster p = "
      f"{excl_cluster_p:.4f}")
print("SURVIVES check 1 (still p<0.05 after excluding Cook's-distance-flagged obs)."
      if SURVIVES_COOKS_CHECK else
      "DOES NOT survive check 1 -- driven by the flagged observation(s), stop here.")


## Step 17.5 — Robustness check 2: wild cluster bootstrap (Rademacher, restricted, B>=999)

Only runs if Step 17.4 survived. With `Company` cluster-robust SEs resting on however many
distinct firms actually appear in this complete-case sample (fewer than 22 in general, since some
firms may be entirely absent after the `SR_CEO_Letter_lag1`/complete-case filtering) -- the
standard concern with few clusters -- the asymptotic cluster-robust p-value can be unreliable.
Wild cluster bootstrap (Cameron/Gelbach/Miller, restricted/null-imposed version, Rademacher
weights, B=999+ replications) is the standard fix: residuals from the model re-fit under
H0: beta=0 are re-weighted by a firm-level +-1 draw each replication, the outcome is
reconstructed, the full model is refit, and the bootstrap p-value is the share of replications
whose |t-statistic| exceeds the original.


In [ ]:
# ============================================================
# Step 17.5 -- Wild cluster bootstrap (Rademacher, restricted, B=999)
# on SR_CEO_Letter_lag1:Sentiment, only if Step 17.4 survived.
# ============================================================
if not SURVIVES_COOKS_CHECK:
    print("Step 17.4 did not survive -- skipping the wild cluster bootstrap per instruction.")
else:
    import patsy
    import statsmodels.api as sm

    formula_rhs_17 = ("Specificity + Sentiment + SR_CEO_Letter_lag1:Specificity + "
                       "SR_CEO_Letter_lag1:Sentiment + Leverage + Size + ROA + Revenue_Growth "
                       "+ C(Company) + C(Q('Fiscal Year'))")
    y_17, X_17_df = patsy.dmatrices(f"Q('CAR[-1,+1]') ~ {formula_rhs_17}", data=part7_sample,
                                     return_type="dataframe")
    y_17 = y_17.iloc[:, 0].values
    target_col_17 = "SR_CEO_Letter_lag1:Sentiment"
    target_idx_17 = list(X_17_df.columns).index(target_col_17)
    X_17 = X_17_df.values

    groups_17 = part7_sample["Company"].values
    clusters_17 = np.unique(groups_17)
    n_clusters_17 = len(clusters_17)
    print(f"Firms (clusters) present in this complete-case sample: {n_clusters_17} of 22")

    m_orig_17 = sm.OLS(y_17, X_17).fit(cov_type="cluster", cov_kwds={"groups": groups_17})
    t_orig_17 = m_orig_17.tvalues[target_idx_17]

    X_r_17 = np.delete(X_17, target_idx_17, axis=1)
    m_r_17 = sm.OLS(y_17, X_r_17).fit()
    yhat_r_17 = m_r_17.fittedvalues
    resid_r_17 = y_17 - yhat_r_17

    B_17 = 999
    rng_17 = np.random.default_rng(20260921)
    t_boot_17 = np.empty(B_17)
    for b in range(B_17):
        w_17 = rng_17.choice([-1.0, 1.0], size=n_clusters_17)
        weight_map_17 = dict(zip(clusters_17, w_17))
        obs_w_17 = np.array([weight_map_17[g] for g in groups_17])
        y_b_17 = yhat_r_17 + resid_r_17 * obs_w_17
        m_b_17 = sm.OLS(y_b_17, X_17).fit(cov_type="cluster", cov_kwds={"groups": groups_17})
        t_boot_17[b] = m_b_17.tvalues[target_idx_17]

    p_wcr_17 = float(np.mean(np.abs(t_boot_17) >= np.abs(t_orig_17)))

    print(f"\nOriginal cluster-robust (asymptotic) p = {m_orig_17.pvalues[target_idx_17]:.4f}")
    print(f"Wild cluster bootstrap p (Rademacher, restricted, B={B_17}) = {p_wcr_17:.4f}")

    SURVIVES_BOOTSTRAP = p_wcr_17 < 0.05
    print("\nSURVIVES check 2 (wild cluster bootstrap p < 0.05)."
          if SURVIVES_BOOTSTRAP else
          "DOES NOT survive check 2 -- the asymptotic cluster p was unreliable at this cluster "
          "count.")

    rule_70_17 = "=" * 70
    print(f"\n{rule_70_17}\nBOTTOM LINE -- SR_CEO_Letter_lag1:Sentiment, both robustness checks\n{rule_70_17}")
    if SURVIVES_COOKS_CHECK and SURVIVES_BOOTSTRAP:
        print("Survives both the Cook's-distance exclusion check and the wild cluster bootstrap. "
              "Still only a flagged candidate given the large MDE noted in Step 17.2 -- not "
              "independent confirmation of effect size, just evidence this particular p-value is "
              "not an artifact of outliers or too-few clusters.")
    else:
        print("Does not survive both checks -- do not treat this coefficient as more than a "
              "flagged candidate that did not hold up under closer scrutiny.")


## Step 17.6 — ESGSentiment substituted for Sentiment: is the standalone-SR effect ESG-specific or general-tone?

Same structure as Step 9.4.2's earlier ESGSentiment substitution check, applied to this section's
interaction test: `SR_CEO_Letter_lag1:ESGSentiment` in place of `SR_CEO_Letter_lag1:Sentiment`,
same complete-case filter logic, controls, FE, HC3 and cluster SEs. Report-only, does not modify
Step 17.1-17.5.


In [ ]:
# ============================================================
# Step 17.6 -- Re-run the SR interaction test with ESGSentiment in
# place of Sentiment.
# ============================================================
part7b_vars = ["CAR[-1,+1]", "Specificity", "ESGSentiment", "SR_CEO_Letter_lag1", "Leverage",
               "Size", "ROA", "Revenue_Growth", "Company", "Fiscal Year"]
part7b_sample = panel14.dropna(subset=part7b_vars).copy()

n_total_b = len(part7b_sample)
n1_b = int((part7b_sample["SR_CEO_Letter_lag1"] == 1).sum())
n0_b = int((part7b_sample["SR_CEO_Letter_lag1"] == 0).sum())
print(f"Complete-case N = {n_total_b} (SR split: {n1_b} vs {n0_b})")

sr_esg_formula = ("Q('CAR[-1,+1]') ~ Specificity + ESGSentiment "
                   "+ SR_CEO_Letter_lag1:Specificity + SR_CEO_Letter_lag1:ESGSentiment "
                   "+ Leverage + Size + ROA + Revenue_Growth "
                   "+ C(Company) + C(Q('Fiscal Year'))")

m_sr_esg_hc3 = smf.ols(formula=sr_esg_formula, data=part7b_sample).fit(cov_type="HC3")
clustered_groups_176 = part7b_sample.loc[m_sr_esg_hc3.model.data.row_labels, "Company"]
m_sr_esg_cluster = smf.ols(formula=sr_esg_formula, data=part7b_sample).fit(
    cov_type="cluster", cov_kwds={"groups": clustered_groups_176}
)

sr_esg_terms = ["Specificity", "ESGSentiment", "SR_CEO_Letter_lag1:Specificity",
                "SR_CEO_Letter_lag1:ESGSentiment"]
sr_esg_rows = []
for term in sr_esg_terms:
    sr_esg_rows.append({
        "term": term,
        "coef": m_sr_esg_hc3.params[term],
        "hc3_se": m_sr_esg_hc3.bse[term], "hc3_p": m_sr_esg_hc3.pvalues[term],
        "cluster_se": m_sr_esg_cluster.bse[term], "cluster_p": m_sr_esg_cluster.pvalues[term],
        "n": int(m_sr_esg_hc3.nobs),
    })
table_17_6 = pd.DataFrame(sr_esg_rows)
print(f"\nTable 17.6 -- CAR[-1,+1] with ESGSentiment, all four terms")
display(table_17_6.round(4))


## Step 17.7 — Multiple-testing correction (ESGSentiment version)

Same 2-term family convention (`SR_CEO_Letter_lag1:Specificity`, `SR_CEO_Letter_lag1:ESGSentiment`),
Holm and BH, raw and corrected p-values side by side.


In [ ]:
# ============================================================
# Step 17.7 -- Holm/BH across the 2 interaction terms (ESGSentiment
# version), both SE types.
# ============================================================
sr_esg_interaction_rows = table_17_6[table_17_6["term"].str.contains("SR_CEO_Letter_lag1")].copy()

reject_holm_hc3_e, p_holm_hc3_e, _, _ = multipletests(sr_esg_interaction_rows["hc3_p"], alpha=0.05, method="holm")
reject_bh_hc3_e, p_bh_hc3_e, _, _ = multipletests(sr_esg_interaction_rows["hc3_p"], alpha=0.05, method="fdr_bh")
reject_holm_cl_e, p_holm_cl_e, _, _ = multipletests(sr_esg_interaction_rows["cluster_p"], alpha=0.05, method="holm")
reject_bh_cl_e, p_bh_cl_e, _, _ = multipletests(sr_esg_interaction_rows["cluster_p"], alpha=0.05, method="fdr_bh")

sr_esg_interaction_rows["hc3_p_holm"] = p_holm_hc3_e
sr_esg_interaction_rows["hc3_sig_holm"] = reject_holm_hc3_e
sr_esg_interaction_rows["hc3_p_bh"] = p_bh_hc3_e
sr_esg_interaction_rows["hc3_sig_bh"] = reject_bh_hc3_e
sr_esg_interaction_rows["cluster_p_holm"] = p_holm_cl_e
sr_esg_interaction_rows["cluster_sig_holm"] = reject_holm_cl_e
sr_esg_interaction_rows["cluster_p_bh"] = p_bh_cl_e
sr_esg_interaction_rows["cluster_sig_bh"] = reject_bh_cl_e

print(f"Table 17.7 -- {len(sr_esg_interaction_rows)}-term family (ESGSentiment version)")
display(sr_esg_interaction_rows[["term", "coef", "hc3_p", "hc3_p_holm", "hc3_sig_holm",
                                  "hc3_p_bh", "hc3_sig_bh", "cluster_p", "cluster_p_holm",
                                  "cluster_sig_holm", "cluster_p_bh", "cluster_sig_bh"]].round(4))


## Step 17.8 — Robustness check 1 (ESGSentiment version): Cook's distance

Same methodology as Step 17.4, applied to `SR_CEO_Letter_lag1:ESGSentiment`.


In [ ]:
# ============================================================
# Step 17.8 -- Cook's distance on the Step 17.6 cluster-SE model;
# refit excluding flagged observations, check whether significance
# survives for SR_CEO_Letter_lag1:ESGSentiment.
# ============================================================
influence_17b = OLSInfluence(m_sr_esg_cluster)
cooks_d_17b = influence_17b.cooks_distance[0]

fit_sample_17b = part7b_sample.loc[m_sr_esg_cluster.model.data.row_labels].copy()
fit_sample_17b["cooks_d"] = cooks_d_17b
threshold_17b = 4 / m_sr_esg_cluster.nobs
flagged_17b = fit_sample_17b[fit_sample_17b["cooks_d"] > threshold_17b]

print(f"Cook's distance threshold (4/N): {threshold_17b:.4f}")
print(f"Flagged observations: {len(flagged_17b)} of {int(m_sr_esg_cluster.nobs)}")
print(flagged_17b[["Company", "Fiscal Year", "cooks_d"]].sort_values("cooks_d", ascending=False).to_string(index=False))

excl_idx_17b = flagged_17b.index
sub_excl_17b = part7b_sample.drop(index=[i for i in excl_idx_17b if i in part7b_sample.index])

m_excl_hc3_17b = smf.ols(formula=sr_esg_formula, data=sub_excl_17b).fit(cov_type="HC3")
m_excl_cluster_17b = smf.ols(formula=sr_esg_formula, data=sub_excl_17b).fit(
    cov_type="cluster", cov_kwds={"groups": sub_excl_17b["Company"]}
)

print(f"\nExcluding {len(flagged_17b)} flagged obs: N={int(m_excl_hc3_17b.nobs)}")
for term in ["SR_CEO_Letter_lag1:Specificity", "SR_CEO_Letter_lag1:ESGSentiment"]:
    print(f"  {term}: coef={m_excl_hc3_17b.params[term]:.4f}  "
          f"HC3 p={m_excl_hc3_17b.pvalues[term]:.4f}  cluster p={m_excl_cluster_17b.pvalues[term]:.4f}")

original_cluster_p_b = m_sr_esg_cluster.pvalues["SR_CEO_Letter_lag1:ESGSentiment"]
excl_cluster_p_b = m_excl_cluster_17b.pvalues["SR_CEO_Letter_lag1:ESGSentiment"]
SURVIVES_COOKS_CHECK_B = excl_cluster_p_b < 0.05

print(f"\nOriginal cluster p = {original_cluster_p_b:.4f}; excluding flagged obs, cluster p = "
      f"{excl_cluster_p_b:.4f}")
print("SURVIVES check 1 (still p<0.05 after excluding Cook's-distance-flagged obs)."
      if SURVIVES_COOKS_CHECK_B else
      "DOES NOT survive check 1 -- driven by the flagged observation(s), stop here.")


## Step 17.9 — Robustness check 2 (ESGSentiment version): wild cluster bootstrap

Same wild cluster bootstrap methodology as Step 17.5 (Rademacher, restricted/null-imposed,
B=999), applied to `SR_CEO_Letter_lag1:ESGSentiment`. Only runs if Step 17.8 survived.


In [ ]:
# ============================================================
# Step 17.9 -- Wild cluster bootstrap (Rademacher, restricted, B=999)
# on SR_CEO_Letter_lag1:ESGSentiment, only if Step 17.8 survived.
# ============================================================
if not SURVIVES_COOKS_CHECK_B:
    print("Step 17.8 did not survive -- skipping the wild cluster bootstrap per instruction.")
else:
    import patsy
    import statsmodels.api as sm

    formula_rhs_17b = ("Specificity + ESGSentiment + SR_CEO_Letter_lag1:Specificity + "
                        "SR_CEO_Letter_lag1:ESGSentiment + Leverage + Size + ROA + Revenue_Growth "
                        "+ C(Company) + C(Q('Fiscal Year'))")
    y_17b, X_17b_df = patsy.dmatrices(f"Q('CAR[-1,+1]') ~ {formula_rhs_17b}", data=part7b_sample,
                                       return_type="dataframe")
    y_17b = y_17b.iloc[:, 0].values
    target_col_17b = "SR_CEO_Letter_lag1:ESGSentiment"
    target_idx_17b = list(X_17b_df.columns).index(target_col_17b)
    X_17b = X_17b_df.values

    groups_17b = part7b_sample["Company"].values
    clusters_17b = np.unique(groups_17b)
    n_clusters_17b = len(clusters_17b)
    print(f"Firms (clusters) present in this complete-case sample: {n_clusters_17b} of 22")

    m_orig_17b = sm.OLS(y_17b, X_17b).fit(cov_type="cluster", cov_kwds={"groups": groups_17b})
    t_orig_17b = m_orig_17b.tvalues[target_idx_17b]

    X_r_17b = np.delete(X_17b, target_idx_17b, axis=1)
    m_r_17b = sm.OLS(y_17b, X_r_17b).fit()
    yhat_r_17b = m_r_17b.fittedvalues
    resid_r_17b = y_17b - yhat_r_17b

    B_17b = 999
    rng_17b = np.random.default_rng(20260921)
    t_boot_17b = np.empty(B_17b)
    for b in range(B_17b):
        w_17b = rng_17b.choice([-1.0, 1.0], size=n_clusters_17b)
        weight_map_17b = dict(zip(clusters_17b, w_17b))
        obs_w_17b = np.array([weight_map_17b[g] for g in groups_17b])
        y_b_17b = yhat_r_17b + resid_r_17b * obs_w_17b
        m_b_17b = sm.OLS(y_b_17b, X_17b).fit(cov_type="cluster", cov_kwds={"groups": groups_17b})
        t_boot_17b[b] = m_b_17b.tvalues[target_idx_17b]

    p_wcr_17b = float(np.mean(np.abs(t_boot_17b) >= np.abs(t_orig_17b)))

    print(f"\nOriginal cluster-robust (asymptotic) p = {m_orig_17b.pvalues[target_idx_17b]:.4f}")
    print(f"Wild cluster bootstrap p (Rademacher, restricted, B={B_17b}) = {p_wcr_17b:.4f}")

    SURVIVES_BOOTSTRAP_B = p_wcr_17b < 0.05
    print("\nSURVIVES check 2 (wild cluster bootstrap p < 0.05)."
          if SURVIVES_BOOTSTRAP_B else
          "DOES NOT survive check 2 -- the asymptotic cluster p was unreliable at this cluster "
          "count.")

    rule_70_17b = "=" * 70
    print(f"\n{rule_70_17b}\nBOTTOM LINE -- SR_CEO_Letter_lag1:ESGSentiment, both robustness "
          f"checks\n{rule_70_17b}")
    if SURVIVES_COOKS_CHECK_B and SURVIVES_BOOTSTRAP_B:
        print("Survives both the Cook's-distance exclusion check and the wild cluster bootstrap, "
              "same standard applied to the general-Sentiment version in Steps 17.4-17.5.")
    else:
        print("Does not survive both checks -- do not treat this coefficient as more than a "
              "flagged candidate that did not hold up under closer scrutiny.")
